# Notebook 0 — Static Properties of Berman's SAW Model

This notebook tests the formal properties of Mitchell Berman's Simple Additive
Weighting (SAW) model of legal determinacy. The model says that the net legal
force on a question equals the sum of each principle's weight times its
activation:

$$F = \sum_i w_i \cdot a_i$$

A case is determinate when $|F|$ exceeds some threshold $\tau$.

Berman claims three things about this model that we can test computationally:

1. **Rough weights suffice.** Because activation variance dominates, the system
   tolerates substantial uncertainty in weight assignments.
2. **Aggregation delivers more law than validation.** Berman's weighted
   aggregation produces determinate outcomes in cases where Hart's
   consensus-based validation model cannot.
3. **Conflicting principles can coexist.** Unlike Hart's model, principles
   that pull in opposite directions still contribute to a determinate outcome.

Each experiment below tests one of these claims. Every result is reported with
bootstrap confidence intervals and sensitivity analyses across the parameter
space — not as a single point estimate from one configuration.

**Runtime:** approximately 20–30 minutes on Colab CPU with High-RAM.


## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%matplotlib inline
!pip install numpy pandas matplotlib seaborn scipy scikit-learn -q
print('Dependencies installed.')

Dependencies installed.


## Core Model

The implementation below is a direct translation of Berman's SAW formula
(p. 330 of the JESP paper). Two design choices are worth noting.

First, weight and activation are kept separate: weight is context-invariant
importance (how much the legal system cares about this principle in general),
activation is context-variant strength (how strongly this principle applies to
the facts of a particular case). Berman is explicit about this distinction
(p. 329): "Weight may be constant across contexts — though not over time —
while activation is context sensitive."

Second, the determinacy threshold $\tau$ is a free parameter, not fixed at
any particular value. Berman himself says he does not know where thresholds
come from (private correspondence). We therefore test across a range of
thresholds rather than committing to one.

In [3]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams.update({
    'font.size': 10, 'axes.titlesize': 12, 'axes.labelsize': 11,
    'figure.dpi': 100, 'savefig.dpi': 150, 'savefig.bbox': 'tight',
})

# ---------------------------------------------------------------------------
# Data structures
# ---------------------------------------------------------------------------

@dataclass
class Principle:
    name: str
    weight: float          # context-invariant importance, [0, 1]
    description: str = ""
    category: str = "general"

    def __post_init__(self):
        assert 0 <= self.weight <= 1, f"Weight must be in [0,1], got {self.weight}"

@dataclass
class Case:
    name: str
    facts: Dict[str, float]   # principle name → activation in [-1, 1]

# ---------------------------------------------------------------------------
# Berman SAW model
# ---------------------------------------------------------------------------

class BermanModel:
    """
    Net Force = Σ(weight_i × activation_i).

    A case is determinate when |Net Force| ≥ threshold.
    """

    def __init__(self, principles: List[Principle], threshold: float = 0.2):
        self.principles = {p.name: p for p in principles}
        self.threshold = threshold

    def net_force(self, case: Case, weight_noise: float = 0.0) -> float:
        f = 0.0
        for name, activation in case.facts.items():
            if name not in self.principles:
                continue
            w = self.principles[name].weight
            if weight_noise > 0:
                w = np.clip(w + np.random.normal(0, weight_noise), 0, 1)
            f += w * activation
        return f

    def is_determinate(self, case: Case, weight_noise: float = 0.0) -> bool:
        return abs(self.net_force(case, weight_noise)) >= self.threshold


# ---------------------------------------------------------------------------
# Principle-set generators
# ---------------------------------------------------------------------------

def make_principles(n: int, distribution: str, seed: int = 0) -> List[Principle]:
    """
    Generate a principle set.

    distribution:
      'uniform'  — all weights near 0.65 (low variance)
      'moderate'  — weights spread from 0.50 to 0.80
      'high'      — weights spread from 0.30 to 0.95 (Berman's 'rangy' weights)
    """
    rng = np.random.RandomState(seed)
    if distribution == 'uniform':
        weights = rng.uniform(0.60, 0.70, n)
    elif distribution == 'moderate':
        weights = rng.uniform(0.50, 0.80, n)
    elif distribution == 'high':
        weights = rng.uniform(0.30, 0.95, n)
    else:
        raise ValueError(distribution)
    return [Principle(f"P{i}", float(w)) for i, w in enumerate(weights)]


def random_case(principle_names: List[str], rng=None) -> Case:
    if rng is None:
        rng = np.random
    facts = {p: float(rng.uniform(-1, 1)) for p in principle_names}
    return Case("synth", facts)


# ---------------------------------------------------------------------------
# Bootstrap helper
# ---------------------------------------------------------------------------

def bootstrap_ci(values, n_boot=2000, ci=95):
    """Return (mean, lo, hi) for a bootstrap confidence interval."""
    arr = np.array(values, dtype=float)
    means = np.array([np.mean(np.random.choice(arr, len(arr), replace=True))
                      for _ in range(n_boot)])
    lo = np.percentile(means, (100 - ci) / 2)
    hi = np.percentile(means, 100 - (100 - ci) / 2)
    return float(np.mean(arr)), float(lo), float(hi)


print("Core model ready.")
print(f"  Net Force = Σ(weight × activation)")
print(f"  Determinate when |F| ≥ threshold")

Core model ready.
  Net Force = Σ(weight × activation)
  Determinate when |F| ≥ threshold


### Measurement Approaches Used in This Notebook

Different experiments test different aspects of "robustness." To avoid
confusion, we define three measurement approaches up front and name them.
Each experiment states which measure it uses.

**Measure A — Agreement with unperturbed outcome.** Given a case, compute its
determinacy judgment with true weights (the "ground truth"). Then perturb the
weights and check whether the perturbed judgment matches. Critical uncertainty
σ* is the noise level where agreement drops below a threshold (default 80%).
*Question answered:* "Does noise change the answer?"

**Measure B — Three-way self-consistency.** Under weight noise, classify each
MC sample as 'positive,' 'negative,' or 'indeterminate.' A case is
"robustly determinate" if ≥90% of samples agree on the same classification.
This is more demanding than Measure A because it distinguishes *direction*,
not just determinacy status.
*Question answered:* "Is the noisy model internally stable?"

**Measure C — Accuracy of biased assessment.** A noisy or biased assessor
evaluates a case. Check whether the assessor's determinacy judgment matches
the true (noiseless) judgment, and separately, whether the assessor gets
the *direction* (sign of net force) correct.
*Question answered:* "Does a biased assessor still get the right answer?"

**Measure D — Pairwise judge agreement.** Two judges with different weight
assignments assess the same case. Check whether they agree on determinacy.
Neither judge is designated as "correct."
*Question answered:* "Would two reasonable judges agree?"

Each measure has its place. A reviewer who sees σ* = 0.072 in one experiment
and σ* = 0.244 in another should check which measure was used — the difference
reflects the measurement design, not an inconsistency.

## Experiment 1 — Weight Uncertainty Tolerance

Berman claims that "the net force of principles may well yield rules
determinately even when particular principles' relative context-invariant
weights are highly uncertain" (p. 329). This experiment tests that claim
directly.

**Method.** We take a set of stipulated weights, generate random cases, and
compute whether each case is determinate. Then we perturb the weights with
Gaussian noise at increasing standard deviations (σ) and check whether the
determinacy judgment changes. The *critical uncertainty* σ* is the noise level
at which determinacy drops below a given accuracy threshold (we test 70%, 80%,
and 90%).

**Why a grid, not a single configuration.** The tolerance will depend on how
many principles there are, how spread out their weights are, and what
determinacy threshold we use. Reporting a single number (like ±0.146) is
meaningless without knowing the conditions that produced it. So we sweep:

- **Principle count:** 5, 9, 15
- **Weight distribution:** uniform (low spread), moderate, high spread
- **Determinacy threshold:** 0.05, 0.10, 0.15, 0.20, 0.30

This gives 45 configurations. For each, we run 30 seeds and report the
critical uncertainty with a 95% bootstrap CI.

In [4]:
# EXPERIMENT 1: WEIGHT UNCERTAINTY TOLERANCE
# ============================================================================
#
# PERFORMANCE NOTE: The original version called the inner loop separately for
# each accuracy threshold. This version computes the full determinacy curve
# once per seed and extracts all crossing points from it — cutting runtime
# by roughly 60×.

def compute_determinacy_curve(n_principles, distribution, det_threshold,
                              n_cases=300, n_mc=40,
                              sigma_levels=None, seed=0):
    """
    For one configuration, compute the determinacy-survival curve across
    noise levels. Returns the curve (one value per sigma level).

    For each case, the "true" outcome is computed at sigma=0. Then at each
    sigma, we perturb weights n_mc times and check whether at least 90%
    of perturbations agree with the true outcome. The fraction of cases
    that pass this test is the curve value.
    """
    if sigma_levels is None:
        sigma_levels = np.linspace(0, 0.50, 16)

    rng = np.random.RandomState(seed)
    principles = make_principles(n_principles, distribution, seed=seed)
    names = [p.name for p in principles]
    model = BermanModel(principles, threshold=det_threshold)

    # Pre-generate cases and true outcomes
    cases = [random_case(names, rng) for _ in range(n_cases)]
    true_outcomes = np.array([model.is_determinate(c) for c in cases])

    # Pre-generate all the weight noise we'll need: shape (n_cases, n_mc, n_principles)
    n_p = len(names)
    all_noise = rng.normal(0, 1, size=(n_cases, n_mc, n_p))

    # Build weight array for vectorised computation
    w_arr = np.array([model.principles[n].weight for n in names])
    # Build activation matrix: (n_cases, n_principles)
    a_mat = np.array([[c.facts[n] for n in names] for c in cases])

    curve = []
    for sigma in sigma_levels:
        if sigma == 0:
            curve.append(1.0)
            continue

        # Noisy weights: (n_cases, n_mc, n_principles)
        noisy_w = np.clip(w_arr[np.newaxis, np.newaxis, :] + sigma * all_noise, 0, 1)

        # Net force under each perturbation: (n_cases, n_mc)
        # = sum over principles of noisy_w * activation
        net_forces = np.sum(noisy_w * a_mat[:, np.newaxis, :], axis=2)

        # Determinate? (n_cases, n_mc)
        noisy_det = np.abs(net_forces) >= det_threshold

        # For each case, what fraction of MC samples agree with true outcome?
        agree_frac = np.mean(noisy_det == true_outcomes[:, np.newaxis], axis=1)

        # Case "survives" if 90%+ of perturbations agree
        survived = np.mean(agree_frac >= 0.9)
        curve.append(float(survived))

    return np.array(curve)


def find_crossing(curve, sigma_levels, threshold):
    """Find sigma where curve drops below threshold, with linear interpolation."""
    for j in range(1, len(curve)):
        if curve[j] < threshold and curve[j-1] >= threshold:
            frac = (threshold - curve[j-1]) / (curve[j] - curve[j-1])
            return float(sigma_levels[j-1] + frac * (sigma_levels[j] - sigma_levels[j-1]))
    return float(sigma_levels[-1])


def experiment_1_weight_uncertainty(n_seeds=20):
    """
    Full grid sweep: 3 distributions × 3 principle counts × 5 thresholds.
    For each cell, run n_seeds independent repetitions.
    Compute the curve ONCE per seed; extract crossings for all accuracy levels.
    """
    distributions = ['uniform', 'moderate', 'high']
    n_principles_list = [5, 9, 15]
    det_thresholds = [0.05, 0.10, 0.15, 0.20, 0.30]
    accuracy_levels = [0.70, 0.80, 0.90]
    sigma_levels = np.linspace(0, 0.50, 16)

    print("=" * 70)
    print("EXPERIMENT 1: Weight Uncertainty Tolerance")
    print("  Grid: 3 distributions × 3 principle counts × 5 thresholds")
    print(f"  Seeds per cell: {n_seeds}")
    print(f"  Sigma levels: {len(sigma_levels)}, cases: 300, MC: 40")
    print("=" * 70)

    rows = []
    total = len(distributions) * len(n_principles_list) * len(det_thresholds)
    done = 0

    for dist in distributions:
        for n_p in n_principles_list:
            for tau in det_thresholds:
                # Collect crossings for each accuracy level
                crits = {acc: [] for acc in accuracy_levels}

                for seed in range(n_seeds):
                    curve = compute_determinacy_curve(
                        n_p, dist, tau, sigma_levels=sigma_levels,
                        seed=seed * 100 + n_p)
                    for acc in accuracy_levels:
                        crits[acc].append(find_crossing(curve, sigma_levels, acc))

                for acc in accuracy_levels:
                    mean, lo, hi = bootstrap_ci(crits[acc])
                    rows.append({
                        'distribution': dist, 'n_principles': n_p,
                        'det_threshold': tau, 'accuracy_level': acc,
                        'critical_sigma_mean': mean,
                        'critical_sigma_lo': lo,
                        'critical_sigma_hi': hi,
                    })

                done += 1
                if done % 5 == 0 or done == total:
                    m80 = np.mean(crits[0.80])
                    lo80, hi80 = bootstrap_ci(crits[0.80])[1:]
                    print(f"  [{done:2d}/{total}] dist={dist:8s} n={n_p:2d} "
                          f"τ={tau:.2f}  σ*={m80:.3f} [{lo80:.3f}, {hi80:.3f}]")

    df = pd.DataFrame(rows)
    return df

exp1_df = experiment_1_weight_uncertainty(n_seeds=20)

EXPERIMENT 1: Weight Uncertainty Tolerance
  Grid: 3 distributions × 3 principle counts × 5 thresholds
  Seeds per cell: 20
  Sigma levels: 16, cases: 300, MC: 40
  [ 5/45] dist=uniform  n= 5 τ=0.30  σ*=0.078 [0.074, 0.082]
  [10/45] dist=uniform  n= 9 τ=0.30  σ*=0.073 [0.070, 0.076]
  [15/45] dist=uniform  n=15 τ=0.30  σ*=0.072 [0.069, 0.074]
  [20/45] dist=moderate n= 5 τ=0.30  σ*=0.079 [0.075, 0.083]
  [25/45] dist=moderate n= 9 τ=0.30  σ*=0.074 [0.070, 0.078]
  [30/45] dist=moderate n=15 τ=0.30  σ*=0.074 [0.071, 0.076]
  [35/45] dist=high     n= 5 τ=0.30  σ*=0.082 [0.076, 0.090]
  [40/45] dist=high     n= 9 τ=0.30  σ*=0.073 [0.069, 0.078]
  [45/45] dist=high     n=15 τ=0.30  σ*=0.073 [0.069, 0.077]


In [5]:
# EXPERIMENT 1 — RESULTS AND VISUALISATION
# ============================================================================

def plot_experiment_1(df):
    # Focus on the 80% accuracy level for the main figure
    df80 = df[df['accuracy_level'] == 0.80].copy()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    fig.suptitle('Experiment 1: Weight Uncertainty Tolerance Across Configurations',
                 fontsize=13, fontweight='bold')

    for i, dist in enumerate(['uniform', 'moderate', 'high']):
        ax = axes[i]
        sub = df80[df80['distribution'] == dist]
        for n_p in [5, 9, 15]:
            chunk = sub[sub['n_principles'] == n_p].sort_values('det_threshold')
            ax.errorbar(chunk['det_threshold'], chunk['critical_sigma_mean'],
                        yerr=[chunk['critical_sigma_mean'] - chunk['critical_sigma_lo'],
                              chunk['critical_sigma_hi'] - chunk['critical_sigma_mean']],
                        marker='o', capsize=4, linewidth=2, label=f'{n_p} principles')
        ax.set_xlabel('Determinacy threshold (τ)')
        if i == 0:
            ax.set_ylabel('Critical weight uncertainty (σ*)')
        ax.set_title(f'{dist.capitalize()} weights')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 0.45)

    plt.tight_layout()
    plt.savefig('/content/exp1_weight_uncertainty.png')
    plt.show()

    # Summary statistics
    df80 = df[df['accuracy_level'] == 0.80]
    overall_mean = df80['critical_sigma_mean'].mean()
    overall_min = df80['critical_sigma_mean'].min()
    overall_max = df80['critical_sigma_mean'].max()

    print(f"\nSUMMARY (80% accuracy criterion):")
    print(f"  Across all {len(df80)} configurations:")
    print(f"    Mean critical uncertainty: ±{overall_mean:.3f}")
    print(f"    Range: ±{overall_min:.3f} to ±{overall_max:.3f}")

    # Show sensitivity to accuracy criterion
    print(f"\n  Sensitivity to accuracy criterion:")
    for acc in [0.70, 0.80, 0.90]:
        sub = df[df['accuracy_level'] == acc]
        m = sub['critical_sigma_mean'].mean()
        print(f"    {int(acc*100)}% criterion: mean σ* = ±{m:.3f}")

    # Identify which parameters matter most
    print(f"\n  Which parameter matters most?")
    for var, col in [('Distribution', 'distribution'),
                     ('N principles', 'n_principles'),
                     ('Threshold', 'det_threshold')]:
        groups = df80.groupby(col)['critical_sigma_mean'].mean()
        spread = groups.max() - groups.min()
        print(f"    {var}: spread = {spread:.3f} "
              f"(range: {groups.min():.3f} to {groups.max():.3f})")

plot_experiment_1(exp1_df)
exp1_df.to_csv('/content/exp1_weight_uncertainty.csv', index=False)


SUMMARY (80% accuracy criterion):
  Across all 45 configurations:
    Mean critical uncertainty: ±0.165
    Range: ±0.070 to ±0.500

  Sensitivity to accuracy criterion:
    70% criterion: mean σ* = ±0.261
    80% criterion: mean σ* = ±0.165
    90% criterion: mean σ* = ±0.050

  Which parameter matters most?
    Distribution: spread = 0.004 (range: 0.163 to 0.166)
    N principles: spread = 0.053 (range: 0.140 to 0.193)
    Threshold: spread = 0.380 (range: 0.075 to 0.455)


## Experiment 2 — Activation Assessment

Berman says activation is "determined by facts about the world" (private
correspondence) — textual clarity, welfare consequences, historical practice,
etc. But judges must *assess* those facts, and their assessments may differ.
This experiment asks: how sensitive is the model to disagreement, error, and
indeterminacy in activation values?

Three sub-experiments address three different theories of what activation
assessment involves:

- **2a (Pluralism):** Two judges hold *legitimately different* views about how
  strongly a principle applies. Neither is wrong. We test how much disagreement
  the model tolerates before the two judges reach different determinacy
  judgments.

- **2b (Objectivism with error):** There is a *true* activation level, but
  judges estimate it with error. We test random error (each judge errs
  independently) versus systematic error (all judges sharing a methodology
  err in the same direction). The systematic-vs-random comparison is the key
  finding: it tells us how much methodological homogeneity hurts.

- **2c (Interval model):** Activation is not a point value but an objective
  range — textual clarity might be anywhere from +0.3 to +0.7, and no
  single number is "correct." We test how wide the range can be before
  determinacy collapses.

In [6]:
# EXPERIMENT 2a: PLURALIST DISAGREEMENT
# ============================================================================

def experiment_2a_disagreement(n_seeds=30, n_cases=800, n_principles=9):
    """
    Two judges assess the same case. Each judge's activations are the true
    activations plus independent Gaussian noise (representing legitimate
    interpretive differences). We check whether both judges reach the same
    determinacy judgment.

    Uses the 2-judge pairwise design: more conservative than a panel test,
    and directly tests whether any two reasonable judges would agree.
    """
    print("\n" + "=" * 70)
    print("EXPERIMENT 2a: Pluralist Disagreement Tolerance")
    print("  Two judges, legitimate differences in activation assessment")
    print("=" * 70)

    disagreement_levels = np.linspace(0, 0.80, 25)
    all_curves = []

    for seed in range(n_seeds):
        rng = np.random.RandomState(seed)
        principles = make_principles(n_principles, 'moderate', seed=seed)
        names = [p.name for p in principles]
        model = BermanModel(principles, threshold=0.2)

        curve = []
        for disagreement in disagreement_levels:
            consistent = 0
            for _ in range(n_cases):
                true_acts = {p: rng.uniform(-1, 1) for p in names}
                # Judge A
                acts_a = {p: np.clip(v + rng.normal(0, disagreement), -1, 1)
                          for p, v in true_acts.items()}
                # Judge B
                acts_b = {p: np.clip(v + rng.normal(0, disagreement), -1, 1)
                          for p, v in true_acts.items()}
                det_a = model.is_determinate(Case("a", acts_a))
                det_b = model.is_determinate(Case("b", acts_b))
                if det_a == det_b:
                    consistent += 1
            curve.append(consistent / n_cases)
        all_curves.append(curve)

    # Find critical disagreement per seed, then bootstrap
    crits = []
    for curve in all_curves:
        for j in range(1, len(curve)):
            if curve[j] < 0.80 and curve[j-1] >= 0.80:
                frac = (0.80 - curve[j-1]) / (curve[j] - curve[j-1])
                crits.append(float(disagreement_levels[j-1] +
                                   frac * (disagreement_levels[j] - disagreement_levels[j-1])))
                break
        else:
            crits.append(float(disagreement_levels[-1]))

    mean_c, lo_c, hi_c = bootstrap_ci(crits)
    mean_curve = np.mean(all_curves, axis=0)

    print(f"\n  Critical disagreement: ±{mean_c:.3f} [{lo_c:.3f}, {hi_c:.3f}]")
    print(f"  Interpretation: Judges can differ by up to σ={mean_c:.3f}")
    print(f"  in their activation assessments and the system still")
    print(f"  produces consistent determinacy judgments 80% of the time.")

    return disagreement_levels, all_curves, crits


# EXPERIMENT 2b: SYSTEMATIC VS RANDOM ERROR
# ============================================================================

def experiment_2b_error_types(n_seeds=30, n_cases=800, n_principles=9):
    """
    Objectivist model: there is a true activation, judges estimate with error.

    Random error: each judge's error is independent.
    Systematic error: all judges sharing a methodology err the same way
    (e.g., textualists systematically overweight textual clarity).

    We also sweep the ratio of systematic to random error to show the
    continuous degradation, not just two extremes.
    """
    print("\n" + "=" * 70)
    print("EXPERIMENT 2b: Systematic vs Random Error")
    print("  True activation exists; judges estimate with noise")
    print("=" * 70)

    error_levels = np.linspace(0, 0.50, 25)
    sys_ratios = [0.0, 0.25, 0.50, 0.75, 1.0]  # fraction that is systematic

    rows = []
    for seed in range(n_seeds):
        rng = np.random.RandomState(seed)
        principles = make_principles(n_principles, 'moderate', seed=seed)
        names = [p.name for p in principles]
        model = BermanModel(principles, threshold=0.2)

        for sys_ratio in sys_ratios:
            for err in error_levels:
                correct = 0
                for _ in range(n_cases):
                    true_acts = {p: rng.uniform(-1, 1) for p in names}
                    true_det = model.is_determinate(Case("t", true_acts))

                    # Build noisy activations
                    systematic_shift = {p: rng.normal(0, err * sys_ratio)
                                        for p in names}
                    noisy_acts = {}
                    for p in names:
                        random_part = rng.normal(0, err * (1 - sys_ratio))
                        noisy_acts[p] = np.clip(
                            true_acts[p] + systematic_shift[p] + random_part,
                            -1, 1)

                    noisy_det = model.is_determinate(Case("n", noisy_acts))
                    if noisy_det == true_det:
                        correct += 1

                rows.append({
                    'seed': seed, 'systematic_ratio': sys_ratio,
                    'error_level': err,
                    'accuracy': correct / n_cases,
                })

    df = pd.DataFrame(rows)

    # Report critical error for each systematic ratio
    print(f"\n  Critical error by systematic fraction:")
    for sr in sys_ratios:
        sub = df[df['systematic_ratio'] == sr].groupby('error_level')['accuracy'].mean()
        sub = sub.reset_index()
        below = sub[sub['accuracy'] < 0.80]
        if len(below) > 0:
            crit = below.iloc[0]['error_level']
        else:
            crit = error_levels[-1]
        print(f"    Systematic ratio {sr:.0%}: critical error ±{crit:.3f}")

    return df


# EXPERIMENT 2c: INTERVAL-VALUED ACTIVATIONS
# ============================================================================

def experiment_2c_ranges(n_seeds=30, n_cases=800, n_principles=9):
    """
    What if activation is objectively a range, not a point?

    For each principle in each case, the true activation is an interval
    [a - r, a + r] where r is the range half-width. Two judges each sample
    independently from this interval. We check whether they agree on
    determinacy.

    This is distinct from 2a because here the disagreement is bounded by
    the objective range — judges cannot disagree more than 2r.
    """
    print("\n" + "=" * 70)
    print("EXPERIMENT 2c: Interval-Valued Activations")
    print("  Activation is an objective range, not a point")
    print("=" * 70)

    range_widths = np.linspace(0, 0.50, 25)
    all_curves = []

    for seed in range(n_seeds):
        rng = np.random.RandomState(seed)
        principles = make_principles(n_principles, 'moderate', seed=seed)
        names = [p.name for p in principles]
        model = BermanModel(principles, threshold=0.2)

        curve = []
        for rw in range_widths:
            consistent = 0
            for _ in range(n_cases):
                centers = {p: rng.uniform(-1, 1) for p in names}
                # Two judges sample from the interval
                acts_a = {p: np.clip(centers[p] + rng.uniform(-rw, rw), -1, 1)
                          for p in names}
                acts_b = {p: np.clip(centers[p] + rng.uniform(-rw, rw), -1, 1)
                          for p in names}
                det_a = model.is_determinate(Case("a", acts_a))
                det_b = model.is_determinate(Case("b", acts_b))
                if det_a == det_b:
                    consistent += 1
            curve.append(consistent / n_cases)
        all_curves.append(curve)

    crits = []
    for curve in all_curves:
        for j in range(1, len(curve)):
            if curve[j] < 0.80 and curve[j-1] >= 0.80:
                frac = (0.80 - curve[j-1]) / (curve[j] - curve[j-1])
                crits.append(float(range_widths[j-1] +
                                   frac * (range_widths[j] - range_widths[j-1])))
                break
        else:
            crits.append(float(range_widths[-1]))

    mean_c, lo_c, hi_c = bootstrap_ci(crits)
    print(f"\n  Critical range half-width: ±{mean_c:.3f} [{lo_c:.3f}, {hi_c:.3f}]")

    return range_widths, all_curves, crits


# RUN ALL OF EXPERIMENT 2
# ============================================================================

dis_levels, dis_curves, dis_crits = experiment_2a_disagreement(n_seeds=30)
err_df = experiment_2b_error_types(n_seeds=30)
rng_levels, rng_curves, rng_crits = experiment_2c_ranges(n_seeds=30)


EXPERIMENT 2a: Pluralist Disagreement Tolerance
  Two judges, legitimate differences in activation assessment

  Critical disagreement: ±0.270 [0.241, 0.305]
  Interpretation: Judges can differ by up to σ=0.270
  in their activation assessments and the system still
  produces consistent determinacy judgments 80% of the time.

EXPERIMENT 2b: Systematic vs Random Error
  True activation exists; judges estimate with noise

  Critical error by systematic fraction:
    Systematic ratio 0%: critical error ±0.354
    Systematic ratio 25%: critical error ±0.479
    Systematic ratio 50%: critical error ±0.500
    Systematic ratio 75%: critical error ±0.438
    Systematic ratio 100%: critical error ±0.375

EXPERIMENT 2c: Interval-Valued Activations
  Activation is an objective range, not a point

  Critical range half-width: ±0.363 [0.344, 0.384]


In [7]:
# EXPERIMENT 2 — COMBINED VISUALISATION
# ============================================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Experiment 2: Activation Assessment — Three Models of Judicial Disagreement',
             fontsize=13, fontweight='bold')

# 2a: Disagreement curves
ax = axes[0]
mean_curve = np.mean(dis_curves, axis=0)
lo_curve = np.percentile(dis_curves, 2.5, axis=0)
hi_curve = np.percentile(dis_curves, 97.5, axis=0)
ax.plot(dis_levels, mean_curve, 'b-', linewidth=2)
ax.fill_between(dis_levels, lo_curve, hi_curve, alpha=0.2, color='blue')
ax.axhline(0.80, color='red', linestyle='--', linewidth=1.5, label='80% threshold')
m, lo, hi = bootstrap_ci(dis_crits)
ax.axvline(m, color='green', linestyle=':', linewidth=2,
           label=f'σ* = {m:.3f} [{lo:.3f}, {hi:.3f}]')
ax.set_xlabel('Disagreement (σ)')
ax.set_ylabel('Pairwise consistency')
ax.set_title('2a: Pluralist disagreement')
ax.legend(fontsize=8)
ax.set_ylim(0.5, 1.02)

# 2b: Systematic vs random
ax = axes[1]
for sr in [0.0, 0.25, 0.50, 0.75, 1.0]:
    sub = err_df[err_df['systematic_ratio'] == sr].groupby('error_level')['accuracy'].mean()
    ax.plot(sub.index, sub.values, linewidth=2,
            label=f'{sr:.0%} systematic')
ax.axhline(0.80, color='red', linestyle='--', linewidth=1.5)
ax.set_xlabel('Total error (σ)')
ax.set_ylabel('Accuracy vs true outcome')
ax.set_title('2b: Systematic vs random error')
ax.legend(fontsize=8)
ax.set_ylim(0.5, 1.02)

# 2c: Interval model
ax = axes[2]
mean_curve_r = np.mean(rng_curves, axis=0)
lo_curve_r = np.percentile(rng_curves, 2.5, axis=0)
hi_curve_r = np.percentile(rng_curves, 97.5, axis=0)
ax.plot(rng_levels, mean_curve_r, 'purple', linewidth=2)
ax.fill_between(rng_levels, lo_curve_r, hi_curve_r, alpha=0.2, color='purple')
ax.axhline(0.80, color='red', linestyle='--', linewidth=1.5)
m_r, lo_r, hi_r = bootstrap_ci(rng_crits)
ax.axvline(m_r, color='green', linestyle=':', linewidth=2,
           label=f'r* = {m_r:.3f} [{lo_r:.3f}, {hi_r:.3f}]')
ax.set_xlabel('Range half-width (r)')
ax.set_ylabel('Pairwise consistency')
ax.set_title('2c: Interval-valued activations')
ax.legend(fontsize=8)
ax.set_ylim(0.5, 1.02)

plt.tight_layout()
plt.savefig('/content/exp2_activation_assessment.png')
plt.show()

# Comparative summary
print("\nCOMPARATIVE SUMMARY — Activation Assessment:")
m_d, _, _ = bootstrap_ci(dis_crits)
m_r, _, _ = bootstrap_ci(rng_crits)

# Get critical errors for 0% and 100% systematic from 2b
pure_random = err_df[err_df['systematic_ratio'] == 0.0].groupby('error_level')['accuracy'].mean()
pure_sys = err_df[err_df['systematic_ratio'] == 1.0].groupby('error_level')['accuracy'].mean()
rand_crit = pure_random[pure_random < 0.80].index[0] if any(pure_random < 0.80) else 0.50
sys_crit = pure_sys[pure_sys < 0.80].index[0] if any(pure_sys < 0.80) else 0.50

print(f"  Pluralist disagreement tolerance: ±{m_d:.3f}")
print(f"  Random error tolerance:           ±{rand_crit:.3f}")
print(f"  Systematic error tolerance:       ±{sys_crit:.3f}")
print(f"  Interval range tolerance:         ±{m_r:.3f}")
print(f"\n  Systematic / Random ratio: {sys_crit / rand_crit:.2f}x")
if sys_crit < rand_crit:
    print(f"  → Systematic error is harder to tolerate than random error.")
    print(f"    Implication: methodological diversity on the bench matters.")
    print(f"    A panel of uniform textualists is epistemically worse than")
    print(f"    a mixed panel whose biases partly cancel.")


COMPARATIVE SUMMARY — Activation Assessment:
  Pluralist disagreement tolerance: ±0.270
  Random error tolerance:           ±0.354
  Systematic error tolerance:       ±0.375
  Interval range tolerance:         ±0.363

  Systematic / Random ratio: 1.06x


### Experiment 2b supplement — Methodology-Specific Systematic Error

The experiment above models systematic error as a per-principle Gaussian bias
drawn randomly and held constant. But in legal practice, systematic error is
methodology-specific: textualists tend to overrate textual clarity and underrate
moral principles; purposivists do the reverse. This variant implements that
structure directly and checks whether the systematic/random distinction changes.

In [8]:
# EXPERIMENT 2b SUPPLEMENT: METHODOLOGY-SPECIFIC SYSTEMATIC ERROR
# ============================================================================
# MEASURE C: Accuracy of biased assessment (determinacy AND direction)

def experiment_2b_methodology_error(n_seeds=20, n_cases=600, n_principles=9):
    """
    Systematic error tied to judicial methodology.

    IMPROVEMENT OVER FIRST VERSION:
    1. Extended error range to 0.80 (avoids ceiling at 0.50)
    2. Tracks BOTH determinacy accuracy and direction accuracy
       - Determinacy: does the panel agree the case is determinate?
       - Direction: does the panel get the SIGN of net force right?
       Direction accuracy is the metric that reveals the cost of homogeneity.
    """
    print("\n" + "=" * 70)
    print("EXPERIMENT 2b SUPPLEMENT: Methodology-Specific Systematic Error")
    print("  Measure C: biased assessment accuracy (determinacy + direction)")
    print("=" * 70)

    error_levels = np.linspace(0, 0.80, 20)  # Extended range

    bias_profiles = {
        'textualist':  np.array([+0.7, +0.6, +0.5, +0.4, -0.5, -0.5, -0.4, -0.3, -0.3]),
        'purposivist': np.array([-0.4, -0.5, -0.3, -0.3, +0.7, +0.6, +0.6, -0.2, +0.4]),
        'pragmatist':  np.array([+0.1, -0.1, +0.2, +0.3, +0.1, +0.1, +0.2, +0.3, +0.5]),
        'originalist': np.array([+0.8, +0.7, +0.3, +0.4, -0.6, -0.6, -0.5, +0.5, -0.4]),
    }

    panel_types = {
        'random_error':   None,
        'all_textualist': ['textualist'] * 5,
        'mixed_panel':    ['textualist', 'purposivist', 'pragmatist',
                           'originalist', 'pragmatist'],
    }

    results = {}
    for panel_name, panel_methods in panel_types.items():
        det_curves = []
        dir_curves = []

        for seed in range(n_seeds):
            rng = np.random.RandomState(seed)
            principles = make_principles(n_principles, 'moderate', seed=seed)
            w_arr = np.array([p.weight for p in principles])

            det_curve = []
            dir_curve = []
            for err in error_levels:
                det_correct = 0
                dir_correct = 0
                for _ in range(n_cases):
                    true_acts = rng.uniform(-1, 1, n_principles)
                    true_force = np.dot(w_arr, true_acts)
                    true_det = abs(true_force) >= 0.2
                    true_sign = np.sign(true_force)

                    if panel_methods is None:
                        noisy_acts = np.clip(
                            true_acts + rng.normal(0, err, n_principles), -1, 1)
                        noisy_force = np.dot(w_arr, noisy_acts)
                    else:
                        forces = []
                        for method in panel_methods:
                            bias = bias_profiles[method] * err
                            noise = rng.normal(0, err * 0.3, n_principles)
                            member_acts = np.clip(true_acts + bias + noise, -1, 1)
                            forces.append(np.dot(w_arr, member_acts))
                        noisy_force = np.mean(forces)

                    noisy_det = abs(noisy_force) >= 0.2
                    noisy_sign = np.sign(noisy_force)

                    if noisy_det == true_det:
                        det_correct += 1
                    if noisy_sign == true_sign or true_sign == 0:
                        dir_correct += 1

                det_curve.append(det_correct / n_cases)
                dir_curve.append(dir_correct / n_cases)
            det_curves.append(det_curve)
            dir_curves.append(dir_curve)

        det_crits = [find_crossing(np.array(c), error_levels, 0.80) for c in det_curves]
        dir_crits = [find_crossing(np.array(c), error_levels, 0.80) for c in dir_curves]
        m_det, lo_det, hi_det = bootstrap_ci(det_crits)
        m_dir, lo_dir, hi_dir = bootstrap_ci(dir_crits)

        results[panel_name] = {
            'det_curves': det_curves, 'dir_curves': dir_curves,
            'det_crits': det_crits, 'dir_crits': dir_crits,
            'det_mean': m_det, 'dir_mean': m_dir,
        }
        print(f"\n  {panel_name:20s}:")
        print(f"    Determinacy accuracy σ*: ±{m_det:.3f} [{lo_det:.3f}, {hi_det:.3f}]")
        print(f"    Direction accuracy σ*:   ±{m_dir:.3f} [{lo_dir:.3f}, {hi_dir:.3f}]")

    # The key comparison: direction accuracy
    print(f"\n  KEY COMPARISON — Direction Accuracy:")
    print(f"    (Does the panel get the SIGN of net force right?)")
    for pname in ['random_error', 'all_textualist', 'mixed_panel']:
        r = results[pname]
        m, lo, hi = bootstrap_ci(r['dir_crits'])
        print(f"    {pname:20s}: σ* = ±{m:.3f} [{lo:.3f}, {hi:.3f}]")

    homo_dir = results['all_textualist']['dir_crits']
    mixed_dir = results['mixed_panel']['dir_crits']
    from scipy.stats import wilcoxon
    stat, pval = wilcoxon(homo_dir, mixed_dir)
    diff = np.mean(mixed_dir) - np.mean(homo_dir)
    print(f"\n    Mixed vs homogeneous (direction): Δσ* = {diff:+.3f}, p = {pval:.4f}")

    homo_det = results['all_textualist']['det_crits']
    mixed_det = results['mixed_panel']['det_crits']
    stat2, pval2 = wilcoxon(homo_det, mixed_det)
    diff2 = np.mean(mixed_det) - np.mean(homo_det)
    print(f"    Mixed vs homogeneous (determinacy): Δσ* = {diff2:+.3f}, p = {pval2:.4f}")

    if diff > 0 and pval < 0.05:
        print(f"\n  → Diverse panels get the DIRECTION right more often.")
        print(f"    Homogeneous panels may produce determinate-but-wrong answers.")
        print(f"    This is the accuracy-vs-determinacy distinction: a panel of")
        print(f"    textualists will confidently decide cases, but they will")
        print(f"    systematically err on cases where moral principles dominate.")
    elif diff2 < 0 and pval2 < 0.05:
        print(f"\n  → Homogeneous panels are more determinate, but diverse panels")
        print(f"    are more accurate on direction. Exactly the trade-off Berman's")
        print(f"    theory should worry about.")

    # Visualise
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Methodology-Specific Error: Determinacy vs Direction Accuracy',
                 fontsize=13, fontweight='bold')

    colors = {'random_error': 'gray', 'all_textualist': 'red', 'mixed_panel': 'green'}
    labels = {'random_error': 'Random (no methodology)',
              'all_textualist': 'All textualist', 'mixed_panel': 'Mixed panel'}

    for pname, res in results.items():
        mc = np.mean(res['det_curves'], axis=0)
        ax1.plot(error_levels, mc, color=colors[pname], linewidth=2, label=labels[pname])
        mc_d = np.mean(res['dir_curves'], axis=0)
        ax2.plot(error_levels, mc_d, color=colors[pname], linewidth=2, label=labels[pname])

    ax1.axhline(0.80, color='black', linestyle='--', linewidth=1.5)
    ax1.set_xlabel('Error magnitude')
    ax1.set_ylabel('Determinacy accuracy')
    ax1.set_title('Determinacy: is the case determinate?')
    ax1.legend(fontsize=9)
    ax1.set_ylim(0.3, 1.02)
    ax1.grid(True, alpha=0.3)

    ax2.axhline(0.80, color='black', linestyle='--', linewidth=1.5)
    ax2.set_xlabel('Error magnitude')
    ax2.set_ylabel('Direction accuracy')
    ax2.set_title('Direction: is the sign of F correct?')
    ax2.legend(fontsize=9)
    ax2.set_ylim(0.3, 1.02)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('/content/exp2b_methodology_error.png')
    plt.show()

    return results

exp2b_method_results = experiment_2b_methodology_error()


EXPERIMENT 2b SUPPLEMENT: Methodology-Specific Systematic Error
  Measure C: biased assessment accuracy (determinacy + direction)

  random_error        :
    Determinacy accuracy σ*: ±0.319 [0.297, 0.340]
    Direction accuracy σ*:   ±0.438 [0.427, 0.448]

  all_textualist      :
    Determinacy accuracy σ*: ±0.795 [0.786, 0.800]
    Direction accuracy σ*:   ±0.800 [0.800, 0.800]

  mixed_panel         :
    Determinacy accuracy σ*: ±0.522 [0.498, 0.552]
    Direction accuracy σ*:   ±0.800 [0.800, 0.800]

  KEY COMPARISON — Direction Accuracy:
    (Does the panel get the SIGN of net force right?)
    random_error        : σ* = ±0.438 [0.427, 0.448]
    all_textualist      : σ* = ±0.800 [0.800, 0.800]
    mixed_panel         : σ* = ±0.800 [0.800, 0.800]

    Mixed vs homogeneous (direction): Δσ* = +0.000, p = nan
    Mixed vs homogeneous (determinacy): Δσ* = -0.273, p = 0.0000

  → Homogeneous panels are more determinate, but diverse panels
    are more accurate on direction. Exactly 

## Experiment 3 — Conflicting Principles Can Coexist

On Hart's picture, law requires consensus: if principles point in opposite
directions, the case is indeterminate. On Berman's picture, conflicting
principles still contribute to a determinate outcome through weighted
aggregation — the stronger side wins, and the magnitude of the net force
tells you how determinate the result is.

This experiment generates cases where principles are deliberately set in
opposition (half positive, half negative) and checks whether Berman's model
still produces determinate outcomes. The finding is that it does — and at a
surprisingly high rate — because the *weighted sum* of opposing forces rarely
lands exactly at zero.

In [9]:
# EXPERIMENT 3: CONFLICTING PRINCIPLES
# ============================================================================

def experiment_3_conflicts(n_seeds=30, n_cases=1000, n_principles=9):
    print("\n" + "=" * 70)
    print("EXPERIMENT 3: Conflicting Principles Can Coexist")
    print("=" * 70)

    case_types = {
        'aligned': lambda rng, n: rng.uniform(0.3, 1.0, n),
        'opposed': lambda rng, n: rng.uniform(-1.0, -0.3, n),
        'balanced_conflict': lambda rng, n: np.concatenate([
            rng.uniform(0.3, 1.0, n // 2),
            rng.uniform(-1.0, -0.3, n - n // 2)
        ]),
        'random': lambda rng, n: rng.uniform(-1.0, 1.0, n),
    }

    rows = []
    for seed in range(n_seeds):
        rng = np.random.RandomState(seed)
        principles = make_principles(n_principles, 'moderate', seed=seed)
        names = [p.name for p in principles]
        model = BermanModel(principles, threshold=0.2)

        for ctype, fn in case_types.items():
            det_count = 0
            forces = []
            for _ in range(n_cases):
                acts = fn(rng, n_principles)
                rng.shuffle(acts)
                case = Case(ctype, dict(zip(names, acts)))
                det = model.is_determinate(case)
                det_count += int(det)
                forces.append(model.net_force(case))
            rows.append({
                'seed': seed, 'case_type': ctype,
                'determinacy_rate': det_count / n_cases,
                'mean_abs_force': float(np.mean(np.abs(forces))),
            })

    df = pd.DataFrame(rows)

    print(f"\n  Determinacy rates (mean ± 95% CI across {n_seeds} seeds):")
    for ctype in case_types:
        vals = df[df['case_type'] == ctype]['determinacy_rate'].values
        m, lo, hi = bootstrap_ci(vals)
        print(f"    {ctype:20s}: {m:.1%} [{lo:.1%}, {hi:.1%}]")

    # The key result: balanced conflicts are still mostly determinate
    bal = df[df['case_type'] == 'balanced_conflict']['determinacy_rate'].values
    m_b, lo_b, hi_b = bootstrap_ci(bal)
    print(f"\n  KEY FINDING: Even when principles directly conflict,")
    print(f"  {m_b:.1%} [{lo_b:.1%}, {hi_b:.1%}] of cases remain determinate.")
    print(f"  This is central to Berman's critique of Hart: consensus")
    print(f"  is not required for determinacy.")

    return df

exp3_df = experiment_3_conflicts()
exp3_df.to_csv('/content/exp3_conflicts.csv', index=False)


EXPERIMENT 3: Conflicting Principles Can Coexist

  Determinacy rates (mean ± 95% CI across 30 seeds):
    aligned             : 100.0% [100.0%, 100.0%]
    opposed             : 100.0% [100.0%, 100.0%]
    balanced_conflict   : 76.8% [76.1%, 77.7%]
    random              : 86.2% [85.8%, 86.6%]

  KEY FINDING: Even when principles directly conflict,
  76.8% [76.1%, 77.6%] of cases remain determinate.
  This is central to Berman's critique of Hart: consensus
  is not required for determinacy.


In [10]:
# EXPERIMENT 3 — VISUALISATION
# ============================================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Experiment 3: Conflicting Principles Can Coexist',
             fontsize=13, fontweight='bold')

# Left: Determinacy rates by case type
case_types = ['aligned', 'opposed', 'balanced_conflict', 'random']
means = []; ci_los = []; ci_his = []
for ct in case_types:
    vals = exp3_df[exp3_df['case_type'] == ct]['determinacy_rate'].values
    m, lo, hi = bootstrap_ci(vals)
    means.append(m); ci_los.append(m - lo); ci_his.append(hi - m)

colors_ct = ['#2ca02c', '#2ca02c', '#F18F01', '#2E86AB']
ax1.bar(range(len(case_types)), means, yerr=[ci_los, ci_his],
        color=colors_ct, alpha=0.8, edgecolor='black', capsize=5)
ax1.set_xticks(range(len(case_types)))
ax1.set_xticklabels([ct.replace('_', '\n') for ct in case_types], fontsize=9)
ax1.set_ylabel('Determinacy rate')
ax1.set_title('Determinacy by Conflict Type', fontweight='bold')
ax1.set_ylim(0, 1.1)
ax1.axhline(0.8, color='red', ls='--', lw=1.5, label='80% threshold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3, axis='y')
for j, m in enumerate(means):
    ax1.text(j, m + ci_his[j] + 0.02, f'{m:.1%}', ha='center', fontweight='bold')

# Right: Force distributions
ax = ax2
for ct, color in zip(['balanced_conflict', 'random'],
                     ['#F18F01', '#2E86AB']):
    forces = exp3_df[exp3_df['case_type'] == ct]['mean_abs_force'].values
    ax.hist(forces, bins=15, alpha=0.6, color=color, edgecolor='black',
            label=ct.replace('_', ' '))
ax.set_xlabel('Mean |net force|')
ax.set_ylabel('Count (seeds)')
ax.set_title('Force Distribution Under Conflict', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/exp3_conflicts.png')
plt.show()

## Experiment 4 — Berman vs Hart

Berman argues that Hart's model delivers "too little law" because it requires
consensus among principles for determinacy. This experiment compares the two
directly.

The obvious reviewer objection is that any Hart implementation is a straw man.
To pre-empt this, we test *two* Hart models and *sweep* the strictness
parameter:

1. **Consensus Hart:** A case is determinate if at least *k*% of principles
   point in the same direction (positive or negative activation > 0.2 in
   absolute value). We vary *k* from 50% to 90%.

2. **Hierarchical Hart:** Principles are organized into tiers (constitutional
   text > precedent > moral principles). A case is determinate if *any* top-tier
   principle is strongly activated, regardless of lower tiers. This is closer
   to Hart's actual rule of recognition, where some criteria take priority.

Berman's model is run unchanged. The comparison is paired by seed, so we can
run a Wilcoxon signed-rank test for significance.

In [11]:
# EXPERIMENT 4: BERMAN VS HART
# ============================================================================

def experiment_4_berman_vs_hart(n_seeds=30, n_cases=1000):
    print("\n" + "=" * 70)
    print("EXPERIMENT 4: Berman vs Hart — Multiple Hart Variants")
    print("=" * 70)

    # Use a realistic 9-principle set with named categories
    def make_named_principles(seed):
        return [
            Principle("text_plain", 0.80, "Plain textual meaning", "textual"),
            Principle("original_meaning", 0.70, "Original meaning", "textual"),
            Principle("precedent", 0.75, "Binding precedent", "institutional"),
            Principle("structure", 0.65, "Structural inference", "textual"),
            Principle("dignity", 0.70, "Human dignity", "moral"),
            Principle("equality", 0.75, "Equal protection", "moral"),
            Principle("liberty", 0.70, "Individual liberty", "moral"),
            Principle("federalism", 0.60, "Federal structure", "structural"),
            Principle("pragmatic", 0.60, "Pragmatic consequences", "moral"),
        ]

    hart_consensus_thresholds = [0.50, 0.60, 0.70, 0.80, 0.90]

    # Hierarchical Hart tiers
    tier1 = {'text_plain', 'original_meaning'}       # constitutional text
    tier2 = {'precedent', 'structure'}                # institutional
    tier3 = {'dignity', 'equality', 'liberty', 'federalism', 'pragmatic'}  # moral/policy

    rows = []
    for seed in range(n_seeds):
        rng = np.random.RandomState(seed)
        principles = make_named_principles(seed)
        names = [p.name for p in principles]
        berman = BermanModel(principles, threshold=0.2)

        berman_det = 0
        hart_results = {k: 0 for k in hart_consensus_thresholds}
        hart_hier_det = 0

        for _ in range(n_cases):
            acts = {p: rng.uniform(-1, 1) for p in names}
            case = Case("test", acts)

            # Berman
            if berman.is_determinate(case):
                berman_det += 1

            # Consensus Hart variants
            positive = sum(1 for a in acts.values() if a > 0.2)
            negative = sum(1 for a in acts.values() if a < -0.2)
            total = len(acts)
            for k in hart_consensus_thresholds:
                if positive / total >= k or negative / total >= k:
                    hart_results[k] += 1

            # Hierarchical Hart: determinate if any tier-1 principle
            # is strongly activated (|a| > 0.7), OR if tier-1 is
            # weakly activated and tier-2 strongly supports
            tier1_strong = any(abs(acts[p]) > 0.7 for p in tier1)
            tier2_strong = any(abs(acts[p]) > 0.7 for p in tier2)
            tier1_moderate = any(abs(acts[p]) > 0.4 for p in tier1)
            if tier1_strong or (tier1_moderate and tier2_strong):
                hart_hier_det += 1

        row = {'seed': seed, 'berman': berman_det / n_cases,
               'hart_hierarchical': hart_hier_det / n_cases}
        for k in hart_consensus_thresholds:
            row[f'hart_consensus_{int(k*100)}'] = hart_results[k] / n_cases
        rows.append(row)

    df = pd.DataFrame(rows)

    # Report with significance tests
    print(f"\n  Berman determinacy: ", end="")
    m, lo, hi = bootstrap_ci(df['berman'].values)
    print(f"{m:.1%} [{lo:.1%}, {hi:.1%}]")

    print(f"\n  Hart variants:")
    for k in hart_consensus_thresholds:
        col = f'hart_consensus_{int(k*100)}'
        m_h, lo_h, hi_h = bootstrap_ci(df[col].values)
        # Paired Wilcoxon test
        stat, pval = stats.wilcoxon(df['berman'].values, df[col].values,
                                     alternative='greater')
        adv = df['berman'].mean() - df[col].mean()
        print(f"    Consensus {int(k*100)}%: {m_h:.1%} [{lo_h:.1%}, {hi_h:.1%}]  "
              f"Berman advantage: +{adv:.1%}  p={pval:.4f}")

    m_hh, lo_hh, hi_hh = bootstrap_ci(df['hart_hierarchical'].values)
    stat, pval = stats.wilcoxon(df['berman'].values,
                                 df['hart_hierarchical'].values,
                                 alternative='greater')
    adv = df['berman'].mean() - df['hart_hierarchical'].mean()
    print(f"    Hierarchical:  {m_hh:.1%} [{lo_hh:.1%}, {hi_hh:.1%}]  "
          f"Berman advantage: +{adv:.1%}  p={pval:.4f}")

    return df

exp4_df = experiment_4_berman_vs_hart()
exp4_df.to_csv('/content/exp4_berman_vs_hart.csv', index=False)


EXPERIMENT 4: Berman vs Hart — Multiple Hart Variants

  Berman determinacy: 86.9% [86.5%, 87.3%]

  Hart variants:
    Consensus 50%: 53.8% [53.2%, 54.4%]  Berman advantage: +33.1%  p=0.0000
    Consensus 60%: 20.1% [19.7%, 20.5%]  Berman advantage: +66.8%  p=0.0000
    Consensus 70%: 4.9% [4.6%, 5.2%]  Berman advantage: +82.0%  p=0.0000
    Consensus 80%: 0.8% [0.6%, 0.9%]  Berman advantage: +86.1%  p=0.0000
    Consensus 90%: 0.1% [0.0%, 0.1%]  Berman advantage: +86.8%  p=0.0000
    Hierarchical:  67.8% [67.3%, 68.4%]  Berman advantage: +19.1%  p=0.0000


In [12]:
# EXPERIMENT 4 — VISUALISATION
# ============================================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Experiment 4: Berman vs Hart — Does Aggregation Deliver More Law?',
             fontsize=13, fontweight='bold')

# Left: Berman advantage vs Hart consensus threshold
ax = ax1
thresholds = [50, 60, 70, 80, 90]
advantages = []
ci_lo = []
ci_hi = []
for k in thresholds:
    col = f'hart_consensus_{k}'
    diff = exp4_df['berman'].values - exp4_df[col].values
    m, lo, hi = bootstrap_ci(diff)
    advantages.append(m)
    ci_lo.append(m - lo)
    ci_hi.append(hi - m)

ax.errorbar(thresholds, advantages, yerr=[ci_lo, ci_hi],
            marker='o', capsize=5, linewidth=2, color='steelblue')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel("Hart consensus threshold (%)")
ax.set_ylabel("Berman advantage (proportion)")
ax.set_title("Berman's advantage across\nHart strictness levels")
ax.grid(True, alpha=0.3)

# Right: Bar chart — Berman vs best Hart variants
ax = ax2
labels = ['Berman']
means = [exp4_df['berman'].mean()]
errs = [[exp4_df['berman'].mean() - bootstrap_ci(exp4_df['berman'])[1]],
        [bootstrap_ci(exp4_df['berman'])[2] - exp4_df['berman'].mean()]]

for k in [60, 70, 80]:
    col = f'hart_consensus_{k}'
    labels.append(f'Hart\n{k}%')
    means.append(exp4_df[col].mean())
    _, lo, hi = bootstrap_ci(exp4_df[col])
    errs[0].append(exp4_df[col].mean() - lo)
    errs[1].append(hi - exp4_df[col].mean())

labels.append('Hart\nhier.')
means.append(exp4_df['hart_hierarchical'].mean())
_, lo, hi = bootstrap_ci(exp4_df['hart_hierarchical'])
errs[0].append(exp4_df['hart_hierarchical'].mean() - lo)
errs[1].append(hi - exp4_df['hart_hierarchical'].mean())

colors = ['#2E86AB'] + ['#F18F01'] * 3 + ['#A23B72']
ax.bar(range(len(labels)), means, yerr=errs, color=colors, alpha=0.8,
       edgecolor='black', capsize=5)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Determinacy rate')
ax.set_title('Determinacy across models')
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/content/exp4_berman_vs_hart.png')
plt.show()

## Experiment 5 — Uncertainty Decomposition

Not all uncertainty is created equal. This experiment separates three sources:

- **Epistemic uncertainty:** We do not know the true weights. This is the same
  as Experiment 1, but placed here for comparison with the other types.

- **Aleatory uncertainty:** Different judges assign different weights. Each
  judge has a stable personal weighting (not random noise each time) that
  departs from the "community" weights. This models inter-judge variation as
  a permanent feature of the bench, not a measurement error.

- **Structural uncertainty:** The SAW formula might be wrong. Maybe forces
  combine non-linearly, or maybe weights act as thresholds rather than
  multipliers. We test SAW against four alternative aggregation functions:
  (a) ordinal (rank-based, ignoring magnitudes), (b) threshold-filtered
  (exclude principles below a weight cutoff, then sum activations only),
  (c) quadratic (w² × a), and (d) square-root (√w × a). If SAW agrees
  with most alternatives in most cases, structural uncertainty is low.

The threshold-filtered alternative is especially important: it tests the
interpretation discussed in conversation with Berman where weight serves
as a gate (above threshold → in; below → out) rather than a multiplier.

In [13]:
# EXPERIMENT 5: UNCERTAINTY DECOMPOSITION
# ============================================================================

def experiment_5_uncertainty(n_seeds=20, n_cases=400, n_principles=9):
    print("\n" + "=" * 70)
    print("EXPERIMENT 5: Uncertainty Decomposition")
    print("  Epistemic, Aleatory, and Structural uncertainty separated")
    print("=" * 70)

    uncertainty_levels = np.linspace(0, 0.50, 16)

    # --- EPISTEMIC (vectorised) ---
    print("\n  Computing epistemic uncertainty...")
    epi_curves = []
    for seed in range(n_seeds):
        rng = np.random.RandomState(seed)
        principles = make_principles(n_principles, 'moderate', seed=seed)
        names = [p.name for p in principles]
        n_p = len(names)
        w_arr = np.array([p.weight for p in principles])

        # Pre-generate cases: (n_cases, n_p)
        a_mat = rng.uniform(-1, 1, size=(n_cases, n_p))
        # True net forces and determinacy
        true_forces = a_mat @ w_arr
        true_det = np.abs(true_forces) >= 0.2

        # Pre-generate noise: (n_cases, n_mc, n_p)
        n_mc = 40
        noise = rng.normal(0, 1, size=(n_cases, n_mc, n_p))

        curve = []
        for unc in uncertainty_levels:
            if unc == 0:
                curve.append(1.0)
                continue
            noisy_w = np.clip(w_arr[None, None, :] + unc * noise, 0, 1)
            noisy_forces = np.sum(noisy_w * a_mat[:, None, :], axis=2)
            noisy_det = np.abs(noisy_forces) >= 0.2
            # For each case, fraction of MC agreeing with true
            agree_frac = np.mean(noisy_det == true_det[:, None], axis=1)
            # Case "survives" if 90%+ agree
            curve.append(float(np.mean(agree_frac >= 0.9)))
        epi_curves.append(curve)

        if (seed + 1) % 10 == 0:
            print(f"    epistemic seed {seed+1}/{n_seeds}")

    # --- ALEATORY ---
    print("  Computing aleatory uncertainty...")
    ale_curves = []
    for seed in range(n_seeds):
        rng = np.random.RandomState(seed)
        principles = make_principles(n_principles, 'moderate', seed=seed)
        names = [p.name for p in principles]
        base_weights = np.array([p.weight for p in principles])

        curve = []
        for unc in uncertainty_levels:
            n_judges = 10
            # Each judge gets STABLE personal weights (drawn once, used for all cases)
            if unc == 0:
                judge_weights = np.tile(base_weights, (n_judges, 1))
            else:
                judge_weights = np.clip(
                    base_weights[None, :] + rng.normal(0, unc, size=(n_judges, n_principles)),
                    0, 1)

            # Generate cases: (n_cases, n_p)
            a_mat = rng.uniform(-1, 1, size=(n_cases, n_principles))

            # Each judge's net force: (n_judges, n_cases)
            forces = judge_weights @ a_mat.T  # (n_judges, n_cases)
            det_per_judge = np.abs(forces) >= 0.2  # (n_judges, n_cases)

            # Fraction of judges agreeing per case
            frac_yes = np.mean(det_per_judge, axis=0)  # (n_cases,)
            # "Determinate" if judges mostly agree (>80% or <20% say yes)
            agreed = np.mean((frac_yes > 0.8) | (frac_yes < 0.2))
            curve.append(float(agreed))
        ale_curves.append(curve)

        if (seed + 1) % 10 == 0:
            print(f"    aleatory seed {seed+1}/{n_seeds}")

    # --- STRUCTURAL ---
    print("  Computing structural uncertainty...")
    struct_rows = []
    for seed in range(n_seeds):
        rng = np.random.RandomState(seed)
        principles = make_principles(n_principles, 'moderate', seed=seed)
        names = [p.name for p in principles]
        w = np.array([p.weight for p in principles])

        a_mat = rng.uniform(-1, 1, size=(n_cases, n_principles))
        n_test = n_cases

        # SAW baseline
        f_saw = a_mat @ w
        det_saw = np.abs(f_saw) >= 0.2

        # Ordinal: count signs
        signs = np.sign(a_mat) * (np.abs(a_mat) > 0.1).astype(float)
        f_ord = signs @ np.sign(w)
        det_ord = np.abs(f_ord) >= 2

        # Threshold-filtered: only principles with w > 0.65, sum activations
        mask_tf = (w > 0.65).astype(float)
        f_tf = a_mat @ mask_tf
        det_tf = np.abs(f_tf) >= 0.5

        # Quadratic: w^2 * a
        f_q = a_mat @ (w ** 2)
        det_q = np.abs(f_q) >= 0.15

        # Sqrt: sqrt(w) * a
        f_s = a_mat @ np.sqrt(w)
        det_s = np.abs(f_s) >= 0.3

        for model_name, det_alt in [('ordinal', det_ord),
                                     ('threshold_filtered', det_tf),
                                     ('quadratic', det_q),
                                     ('sqrt', det_s)]:
            agree = float(np.mean(det_saw == det_alt))
            struct_rows.append({'seed': seed, 'alt_model': model_name,
                                'agreement': agree})

    struct_df = pd.DataFrame(struct_rows)

    # Report
    epi_crits = _find_crits(epi_curves, uncertainty_levels, 0.80)
    ale_crits = _find_crits(ale_curves, uncertainty_levels, 0.80)
    m_e, lo_e, hi_e = bootstrap_ci(epi_crits)
    m_a, lo_a, hi_a = bootstrap_ci(ale_crits)

    print(f"\n  Epistemic (weight uncertainty): σ* = ±{m_e:.3f} [{lo_e:.3f}, {hi_e:.3f}]")
    print(f"  Aleatory (judge variation):     σ* = ±{m_a:.3f} [{lo_a:.3f}, {hi_a:.3f}]")
    if m_a > m_e:
        ratio = m_a / m_e if m_e > 0 else float('inf')
        print(f"  → Aleatory {ratio:.1f}× more tolerable than epistemic.")
        print(f"    Judges can disagree about weights more than the system")
        print(f"    can tolerate not knowing the true weights at all.")

    print(f"\n  Structural (model specification):")
    for model_name in ['ordinal', 'threshold_filtered', 'quadratic', 'sqrt']:
        vals = struct_df[struct_df['alt_model'] == model_name]['agreement'].values
        m_s, lo_s, hi_s = bootstrap_ci(vals)
        print(f"    SAW vs {model_name:20s}: {m_s:.1%} [{lo_s:.1%}, {hi_s:.1%}] agreement")

    tf_agree = struct_df[struct_df['alt_model'] == 'threshold_filtered']['agreement'].mean()
    print(f"\n  NOTE: SAW vs threshold-filtered agreement = {tf_agree:.1%}.")
    print(f"  This tests the alternative interpretation where weight is a gate,")
    print(f"  not a multiplier. {'High' if tf_agree > 0.85 else 'Moderate'} agreement suggests")
    print(f"  the two interpretations {'mostly converge' if tf_agree > 0.85 else 'diverge meaningfully'}.")

    return epi_curves, ale_curves, struct_df, epi_crits, ale_crits


def _find_crits(curves, levels, threshold):
    crits = []
    for curve in curves:
        found = False
        for j in range(1, len(curve)):
            if curve[j] < threshold and curve[j-1] >= threshold:
                frac = (threshold - curve[j-1]) / (curve[j] - curve[j-1])
                crits.append(float(levels[j-1] + frac * (levels[j] - levels[j-1])))
                found = True
                break
        if not found:
            crits.append(float(levels[-1]))
    return crits


epi_curves, ale_curves, struct_df, epi_crits, ale_crits = experiment_5_uncertainty()
struct_df.to_csv('/content/exp5_structural.csv', index=False)


EXPERIMENT 5: Uncertainty Decomposition
  Epistemic, Aleatory, and Structural uncertainty separated

  Computing epistemic uncertainty...
    epistemic seed 10/20
    epistemic seed 20/20
  Computing aleatory uncertainty...
    aleatory seed 10/20
    aleatory seed 20/20
  Computing structural uncertainty...

  Epistemic (weight uncertainty): σ* = ±0.072 [0.067, 0.077]
  Aleatory (judge variation):     σ* = ±0.086 [0.082, 0.091]
  → Aleatory 1.2× more tolerable than epistemic.
    Judges can disagree about weights more than the system
    can tolerate not knowing the true weights at all.

  Structural (model specification):
    SAW vs ordinal             : 66.6% [65.5%, 67.8%] agreement
    SAW vs threshold_filtered  : 66.3% [62.5%, 69.8%] agreement
    SAW vs quadratic           : 91.7% [90.8%, 92.5%] agreement
    SAW vs sqrt                : 95.6% [95.1%, 96.1%] agreement

  NOTE: SAW vs threshold-filtered agreement = 66.3%.
  This tests the alternative interpretation where weight 

In [14]:
# EXPERIMENT 5 — VISUALISATION
# ============================================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Experiment 5: Uncertainty Decomposition',
             fontsize=13, fontweight='bold')

# Left: Epistemic vs Aleatory curves
ax = axes[0]
epi_mean = np.mean(epi_curves, axis=0)
ale_mean = np.mean(ale_curves, axis=0)
epi_lo = np.percentile(epi_curves, 2.5, axis=0)
epi_hi = np.percentile(epi_curves, 97.5, axis=0)
ale_lo = np.percentile(ale_curves, 2.5, axis=0)
ale_hi = np.percentile(ale_curves, 97.5, axis=0)
unc_levels = np.linspace(0, 0.50, 16)

ax.plot(unc_levels, epi_mean, 'b-', lw=2, label='Epistemic (unknown weights)')
ax.fill_between(unc_levels, epi_lo, epi_hi, alpha=0.15, color='blue')
ax.plot(unc_levels, ale_mean, 'g-', lw=2, label='Aleatory (judge variation)')
ax.fill_between(unc_levels, ale_lo, ale_hi, alpha=0.15, color='green')
ax.axhline(0.80, color='red', ls='--', lw=1.5)
ax.set_xlabel('Uncertainty (σ)')
ax.set_ylabel('Determinacy survival / Agreement')
ax.set_title('Epistemic vs Aleatory', fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0.4, 1.02)
ax.grid(True, alpha=0.3)

# Middle: Critical uncertainty comparison
ax = axes[1]
m_e, _, _ = bootstrap_ci(epi_crits)
m_a, _, _ = bootstrap_ci(ale_crits)
ax.bar([0, 1], [m_e, m_a], color=['steelblue', 'green'],
       alpha=0.8, edgecolor='black')
ax.set_xticks([0, 1])
ax.set_xticklabels(['Epistemic', 'Aleatory'])
ax.set_ylabel('Critical uncertainty (σ*)')
ax.set_title('Which Is More Tolerable?', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Right: Structural agreement (uncalibrated)
ax = axes[2]
models = ['ordinal', 'threshold_filtered', 'quadratic', 'sqrt']
agrees = []
for m in models:
    vals = struct_df[struct_df['alt_model'] == m]['agreement'].values
    agrees.append(np.mean(vals))
colors_s = ['coral' if a < 0.75 else 'gold' if a < 0.90 else '#2ca02c' for a in agrees]
ax.barh(range(len(models)), agrees, color=colors_s, alpha=0.8, edgecolor='black')
ax.set_yticks(range(len(models)))
ax.set_yticklabels([m.replace('_', '\n') for m in models], fontsize=9)
ax.set_xlabel('Agreement with SAW')
ax.set_title('Structural: SAW vs Alternatives\n(uncalibrated)', fontweight='bold')
ax.set_xlim(0, 1.05)
ax.axvline(0.80, color='red', ls='--', lw=1.5)
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('/content/exp5_uncertainty.png')
plt.show()

### Experiment 5 supplement — Aleatory (Pairwise) and Structural (Calibrated)

Two additional measurements for completeness:

**Aleatory (pairwise):** The main Experiment 5 uses a 10-judge panel. This
supplement uses the 2-judge pairwise design from Experiment 2a, which is more
conservative. If the epistemic/aleatory ratio changes, both designs are
reported and the reader can see why.

**Structural (calibrated):** The main Experiment 5 compares SAW to alternative
models using hardcoded thresholds for each alternative. This supplement
calibrates each alternative's threshold so that it produces the same baseline
determinacy rate as SAW. The agreement rate then measures whether the models
agree on *which cases* are determinate, not just *how many*.

In [15]:
# EXPERIMENT 5 SUPPLEMENT: PAIRWISE ALEATORY + CALIBRATED STRUCTURAL
# ============================================================================

def experiment_5_aleatory_pairwise(n_seeds=20, n_cases=400, n_principles=9):
    """
    Aleatory uncertainty measured via 2-judge pairwise comparison.
    Each judge has stable personal weights (shifted from community weights).
    Check whether both judges agree on determinacy for each case.
    """
    print("\n" + "=" * 70)
    print("EXPERIMENT 5 SUPPLEMENT: Aleatory Uncertainty (Pairwise Design)")
    print("=" * 70)

    uncertainty_levels = np.linspace(0, 0.50, 16)
    all_curves = []

    for seed in range(n_seeds):
        rng = np.random.RandomState(seed)
        principles = make_principles(n_principles, 'moderate', seed=seed)
        w_arr = np.array([p.weight for p in principles])
        names = [p.name for p in principles]

        curve = []
        for unc in uncertainty_levels:
            if unc == 0:
                curve.append(1.0)
                continue
            # Two judges with stable personal weight shifts
            judge_a_w = np.clip(w_arr + rng.normal(0, unc, n_principles), 0, 1)
            judge_b_w = np.clip(w_arr + rng.normal(0, unc, n_principles), 0, 1)

            consistent = 0
            for _ in range(n_cases):
                acts = rng.uniform(-1, 1, n_principles)
                det_a = abs(np.dot(judge_a_w, acts)) >= 0.2
                det_b = abs(np.dot(judge_b_w, acts)) >= 0.2
                if det_a == det_b:
                    consistent += 1
            curve.append(consistent / n_cases)
        all_curves.append(curve)

    crits = [find_crossing(np.array(c), uncertainty_levels, 0.80) for c in all_curves]
    m, lo, hi = bootstrap_ci(crits)

    # Compare with main Experiment 5 aleatory result
    print(f"\n  Pairwise aleatory:   σ* = ±{m:.3f} [{lo:.3f}, {hi:.3f}]")
    print(f"  (Main Exp 5 panel:   σ* = ±{bootstrap_ci(ale_crits)[0]:.3f})")
    print(f"  (Main Exp 5 epistemic: σ* = ±{bootstrap_ci(epi_crits)[0]:.3f})")

    pairwise_ratio = m / bootstrap_ci(epi_crits)[0] if bootstrap_ci(epi_crits)[0] > 0 else float('inf')
    panel_ratio = bootstrap_ci(ale_crits)[0] / bootstrap_ci(epi_crits)[0] if bootstrap_ci(epi_crits)[0] > 0 else float('inf')

    print(f"\n  Aleatory/Epistemic ratio (pairwise): {pairwise_ratio:.2f}×")
    print(f"  Aleatory/Epistemic ratio (panel):    {panel_ratio:.2f}×")
    print(f"  Both designs show aleatory {'more' if m > bootstrap_ci(epi_crits)[0] else 'less'} tolerable than epistemic.")

    return crits


def experiment_5_structural_calibrated(n_seeds=20, n_cases=600, n_principles=9):
    """
    Calibrated structural comparison: adjust each alternative model's threshold
    so it produces the same baseline determinacy rate as SAW, then compare
    which cases they agree on.
    """
    print("\n" + "=" * 70)
    print("EXPERIMENT 5 SUPPLEMENT: Structural Uncertainty (Calibrated)")
    print("  Each alternative model calibrated to same baseline determinacy as SAW")
    print("=" * 70)

    rows = []
    for seed in range(n_seeds):
        rng = np.random.RandomState(seed)
        principles = make_principles(n_principles, 'moderate', seed=seed)
        w = np.array([p.weight for p in principles])
        a_mat = rng.uniform(-1, 1, size=(n_cases, n_principles))

        # SAW baseline
        f_saw = a_mat @ w
        saw_det_rate = np.mean(np.abs(f_saw) >= 0.2)

        # For each alternative, find threshold that gives same det rate
        alt_models = {
            'ordinal': a_mat @ (np.sign(w) * (np.abs(a_mat) > 0.1).astype(float).T).diagonal()
                       if False else None,  # placeholder — compute below
            'threshold_filtered': None,
            'quadratic': a_mat @ (w ** 2),
            'sqrt': a_mat @ np.sqrt(w),
        }

        # Compute forces for each alternative
        # Ordinal: sum of sign(activation) for active principles, weighted by sign(weight)
        signs = np.sign(a_mat) * (np.abs(a_mat) > 0.1).astype(float)
        f_ordinal = signs @ np.sign(w)

        # Threshold-filtered: sum activations for principles with w > median(w)
        mask_tf = (w > np.median(w)).astype(float)
        f_tf = a_mat @ mask_tf

        # Quadratic and sqrt
        f_quad = a_mat @ (w ** 2)
        f_sqrt = a_mat @ np.sqrt(w)

        alt_forces = {
            'ordinal': f_ordinal,
            'threshold_filtered': f_tf,
            'quadratic': f_quad,
            'sqrt': f_sqrt,
        }

        for model_name, f_alt in alt_forces.items():
            # Calibrate: find threshold for alt model that gives same det rate as SAW
            abs_f = np.abs(f_alt)
            # The threshold that gives det_rate ≈ saw_det_rate is the
            # (1 - saw_det_rate) quantile of |f_alt|
            calibrated_tau = np.percentile(abs_f, (1 - saw_det_rate) * 100)
            det_alt = np.abs(f_alt) >= calibrated_tau
            det_saw = np.abs(f_saw) >= 0.2

            # Agreement: same classification on each case
            agreement = float(np.mean(det_saw == det_alt))
            # Also check: among cases SAW calls determinate, what fraction
            # does the alternative also call determinate?
            saw_pos = det_saw.astype(bool)
            if saw_pos.sum() > 0:
                overlap = float(np.mean(det_alt[saw_pos]))
            else:
                overlap = float('nan')

            rows.append({
                'seed': seed, 'alt_model': model_name,
                'agreement': agreement, 'overlap_given_saw': overlap,
                'calibrated_threshold': float(calibrated_tau),
            })

    df = pd.DataFrame(rows)

    print(f"\n  Calibrated structural agreement (same baseline determinacy rate):")
    for model_name in ['ordinal', 'threshold_filtered', 'quadratic', 'sqrt']:
        sub = df[df['alt_model'] == model_name]
        m_a, lo_a, hi_a = bootstrap_ci(sub['agreement'].values)
        m_o, lo_o, hi_o = bootstrap_ci(sub['overlap_given_saw'].values)
        print(f"    SAW vs {model_name:20s}: "
              f"agreement={m_a:.1%} [{lo_a:.1%}, {hi_a:.1%}]  "
              f"overlap={m_o:.1%} [{lo_o:.1%}, {hi_o:.1%}]")

    # Compare with uncalibrated results from main Exp 5
    print(f"\n  COMPARISON — uncalibrated vs calibrated agreement:")
    for model_name in ['ordinal', 'threshold_filtered', 'quadratic', 'sqrt']:
        uncal = struct_df[struct_df['alt_model'] == model_name]['agreement'].mean()
        cal = df[df['alt_model'] == model_name]['agreement'].mean()
        print(f"    {model_name:20s}: uncalibrated={uncal:.1%}  calibrated={cal:.1%}  "
              f"Δ={cal-uncal:+.1%}")

    tf_cal = df[df['alt_model'] == 'threshold_filtered']['agreement'].mean()
    tf_uncal = struct_df[struct_df['alt_model'] == 'threshold_filtered']['agreement'].mean()
    print(f"\n  KEY: SAW vs threshold-filtered agreement after calibration: {tf_cal:.1%}")
    print(f"  (Was {tf_uncal:.1%} before calibration.)")
    if tf_cal > 0.85:
        print(f"  → After controlling for baseline rate, the two interpretations")
        print(f"    mostly agree. The earlier divergence was partly a threshold artifact.")
    else:
        print(f"  → Even after calibration, genuine divergence remains ({1-tf_cal:.1%} of cases).")
        print(f"    The 'weight as gate' and 'weight as multiplier' interpretations")
        print(f"    produce different answers for a meaningful fraction of cases.")

    return df

exp5_ale_pairwise_crits = experiment_5_aleatory_pairwise()
exp5_struct_calibrated_df = experiment_5_structural_calibrated()
exp5_struct_calibrated_df.to_csv('/content/exp5_structural_calibrated.csv', index=False)


EXPERIMENT 5 SUPPLEMENT: Aleatory Uncertainty (Pairwise Design)

  Pairwise aleatory:   σ* = ±0.244 [0.213, 0.278]
  (Main Exp 5 panel:   σ* = ±0.086)
  (Main Exp 5 epistemic: σ* = ±0.072)

  Aleatory/Epistemic ratio (pairwise): 3.38×
  Aleatory/Epistemic ratio (panel):    1.19×
  Both designs show aleatory more tolerable than epistemic.

EXPERIMENT 5 SUPPLEMENT: Structural Uncertainty (Calibrated)
  Each alternative model calibrated to same baseline determinacy as SAW

  Calibrated structural agreement (same baseline determinacy rate):
    SAW vs ordinal             : agreement=81.7% [80.7%, 83.0%]  overlap=91.2% [90.3%, 92.4%]
    SAW vs threshold_filtered  : agreement=77.3% [76.0%, 78.6%]  overlap=86.7% [85.9%, 87.6%]
    SAW vs quadratic           : agreement=92.4% [91.5%, 93.3%]  overlap=95.6% [95.1%, 96.1%]
    SAW vs sqrt                : agreement=96.3% [95.9%, 96.8%]  overlap=97.9% [97.6%, 98.1%]

  COMPARISON — uncalibrated vs calibrated agreement:
    ordinal             : 

In [16]:
# EXPERIMENT 5 SUPPLEMENT — VISUALISATION
# ============================================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Experiment 5 Supplement: Calibrated Structural + Pairwise Aleatory',
             fontsize=13, fontweight='bold')

# Left: Calibrated vs uncalibrated agreement
models = ['ordinal', 'threshold_filtered', 'quadratic', 'sqrt']
uncal = [struct_df[struct_df['alt_model']==m]['agreement'].mean() for m in models]
cal = [exp5_struct_calibrated_df[exp5_struct_calibrated_df['alt_model']==m]['agreement'].mean()
       for m in models]

x = np.arange(len(models))
w = 0.35
ax1.barh(x + w/2, uncal, w, label='Uncalibrated', color='#aec7e8', edgecolor='black')
ax1.barh(x - w/2, cal, w, label='Calibrated', color='#1f77b4', edgecolor='black')
ax1.set_yticks(x)
ax1.set_yticklabels([m.replace('_', '\n') for m in models], fontsize=9)
ax1.set_xlabel('Agreement with SAW')
ax1.set_title('Structural: Calibration Effect', fontweight='bold')
ax1.legend(fontsize=9)
ax1.set_xlim(0, 1.05)
ax1.grid(True, alpha=0.3, axis='x')

# Right: Aleatory comparison (panel vs pairwise)
m_panel, _, _ = bootstrap_ci(ale_crits)
m_pair, _, _ = bootstrap_ci(exp5_ale_pairwise_crits)
m_epi, _, _ = bootstrap_ci(epi_crits)
ax2.bar([0, 1, 2], [m_epi, m_panel, m_pair],
        color=['steelblue', 'green', 'limegreen'],
        alpha=0.8, edgecolor='black')
ax2.set_xticks([0, 1, 2])
ax2.set_xticklabels(['Epistemic', 'Aleatory\n(10-judge panel)', 'Aleatory\n(pairwise)'],
                     fontsize=9)
ax2.set_ylabel('Critical uncertainty (σ*)')
ax2.set_title('Measurement Design Matters', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/content/exp5_supplement.png')
plt.show()

## Experiment 6 — The Threshold Is Normatively Loaded

There is no objectively correct determinacy threshold. A low threshold (τ = 0.05)
maximises the number of determinate outcomes but risks calling close cases
"determinate" when they are genuinely contested. A high threshold (τ = 0.30)
is more cautious but leaves many cases indeterminate.

This experiment maps the trade-off. For each threshold, we compute:
- **Determinacy rate:** What fraction of cases come out determinate?
- **Stability:** Of the cases declared determinate, how many survive
  a small perturbation to the weights? (This proxies for "false determinacy" —
  cases that are technically above threshold but fragile.)

The finding is not that one threshold is right. The finding is that threshold
selection is a *normative* choice with quantifiable consequences, not a
technical parameter that can be optimised away.

In [17]:
# EXPERIMENT 6: NORMATIVE THRESHOLD MAP
# ============================================================================

def experiment_6_threshold_map(n_seeds=20, n_cases=600, n_principles=9):
    print("\n" + "=" * 70)
    print("EXPERIMENT 6: Normative Threshold Mapping")
    print("  No 'correct' threshold — this maps the trade-off space")
    print("=" * 70)

    thresholds = np.linspace(0.02, 0.40, 20)
    perturbation_sigma = 0.10

    rows = []
    for seed in range(n_seeds):
        rng = np.random.RandomState(seed)
        principles = make_principles(n_principles, 'moderate', seed=seed)
        names = [p.name for p in principles]
        n_p = len(names)
        w_arr = np.array([p.weight for p in principles])

        # Pre-generate all cases and noise
        a_mat = rng.uniform(-1, 1, size=(n_cases, n_p))
        forces = a_mat @ w_arr  # (n_cases,)
        n_mc_stab = 30
        noise = rng.normal(0, perturbation_sigma, size=(n_cases, n_mc_stab, n_p))
        noisy_w = np.clip(w_arr[None, None, :] + noise, 0, 1)
        noisy_forces = np.sum(noisy_w * a_mat[:, None, :], axis=2)  # (n_cases, n_mc)

        for tau in thresholds:
            det_mask = np.abs(forces) >= tau
            det_count = int(det_mask.sum())

            if det_count > 0:
                # Stability: of determinate cases, how many survive perturbation?
                noisy_det = np.abs(noisy_forces) >= tau  # (n_cases, n_mc)
                # For each case, fraction of MC that agree it's determinate
                survive_frac = np.mean(noisy_det, axis=1)  # (n_cases,)
                # Among actually-determinate cases, how many survive at 80%+
                stable = int(np.sum((survive_frac[det_mask] >= 0.8)))
                stability = stable / det_count
            else:
                stability = 0.0

            rows.append({
                'seed': seed, 'threshold': float(tau),
                'determinacy_rate': det_count / n_cases,
                'stability_rate': stability,
            })

        if (seed + 1) % 5 == 0:
            print(f"  seed {seed+1}/{n_seeds}")

    df = pd.DataFrame(rows)

    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Experiment 6: Threshold Is a Normative Choice, Not a Technical Parameter',
                 fontsize=13, fontweight='bold')

    grouped = df.groupby('threshold').agg({
        'determinacy_rate': ['mean', 'std'],
        'stability_rate': ['mean', 'std'],
    })

    ax1.errorbar(thresholds, grouped[('determinacy_rate', 'mean')],
                 yerr=grouped[('determinacy_rate', 'std')],
                 marker='o', capsize=3, linewidth=2, color='steelblue',
                 label='Determinacy rate')
    ax1.errorbar(thresholds, grouped[('stability_rate', 'mean')],
                 yerr=grouped[('stability_rate', 'std')],
                 marker='s', capsize=3, linewidth=2, color='coral',
                 label='Stability rate')
    ax1.set_xlabel('Threshold (τ)')
    ax1.set_ylabel('Rate')
    ax1.set_title('Determinacy vs Stability')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Trade-off curve
    ax2.scatter(grouped[('determinacy_rate', 'mean')],
                grouped[('stability_rate', 'mean')],
                c=thresholds, cmap='viridis', s=80, edgecolor='black')
    ax2.set_xlabel('Determinacy rate')
    ax2.set_ylabel('Stability rate')
    ax2.set_title('The trade-off frontier')
    cb = plt.colorbar(ax2.collections[0], ax=ax2)
    cb.set_label('Threshold (τ)')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('/content/exp6_threshold_map.png')
    plt.show()

    print(f"\n  Trade-off at selected thresholds:")
    for tau in [0.05, 0.10, 0.15, 0.20, 0.30]:
        sub = df[np.isclose(df['threshold'], tau, atol=0.015)]
        if len(sub) > 0:
            det_m = sub['determinacy_rate'].mean()
            stab_m = sub['stability_rate'].mean()
            print(f"    τ={tau:.2f}: {det_m:.1%} determinate, "
                  f"{stab_m:.1%} stable")

    return df

exp6_df = experiment_6_threshold_map()
exp6_df.to_csv('/content/exp6_threshold_map.csv', index=False)


EXPERIMENT 6: Normative Threshold Mapping
  No 'correct' threshold — this maps the trade-off space
  seed 5/20
  seed 10/20
  seed 15/20
  seed 20/20

  Trade-off at selected thresholds:
    τ=0.05: 96.6% determinate, 96.7% stable
    τ=0.10: 92.9% determinate, 91.7% stable
    τ=0.15: 89.3% determinate, 90.3% stable
    τ=0.20: 85.8% determinate, 89.5% stable
    τ=0.30: 78.7% determinate, 89.2% stable


## Experiment 7 — Principle Count: The Determinacy–Robustness Trade-off

Berman does not specify how many principles a legal system should contain.
But the number matters. More principles mean more information entering the
aggregation, which pushes net forces further from zero and increases
determinacy. At the same time, more principles mean more weights that must
be approximately correct, which decreases robustness to uncertainty.

This is a genuine trade-off, not a parameter to optimise. A legal system with
3 principles is robust but frequently indeterminate. A system with 25 is
almost always determinate but fragile to weight misspecification. The
experiment maps this frontier.

We also test Berman's specific claim (p. 329) that "the net force of
principles may well yield rules determinately even when particular principles'
relative context-invariant weights are highly uncertain." If the trade-off
is steep — if adding principles rapidly degrades robustness — then Berman's
claim holds only for small principle sets. If the trade-off is gentle,
the claim is more general.

In [18]:
# EXPERIMENT 7: PRINCIPLE COUNT TRADE-OFF
# ============================================================================

def experiment_7_principle_count(n_seeds=20, n_cases=400, n_mc=40):
    """
    Sweep principle count from 3 to 25. For each count, measure:
    - Baseline determinacy rate (no uncertainty)
    - Critical weight uncertainty (where determinacy survival drops below 80%)
    - The product (determinacy × robustness) as a composite "quality" score

    Uses moderate weight distribution and threshold 0.2 as the reference
    configuration, matching Experiment 1's central case.
    """
    print("\n" + "=" * 70)
    print("EXPERIMENT 7: Principle Count — Determinacy vs Robustness")
    print("  How many principles can Berman's model handle?")
    print("=" * 70)

    principle_counts = [3, 5, 7, 9, 12, 15, 20, 25]
    sigma_levels = np.linspace(0, 0.50, 16)
    det_threshold = 0.2

    rows = []
    for n_p in principle_counts:
        seed_dets = []
        seed_crits = []

        for seed in range(n_seeds):
            rng = np.random.RandomState(seed * 100 + n_p)
            principles = make_principles(n_p, 'moderate', seed=seed)
            names = [p.name for p in principles]
            w_arr = np.array([p.weight for p in principles])

            # Pre-generate cases and noise
            a_mat = rng.uniform(-1, 1, size=(n_cases, n_p))
            noise = rng.normal(0, 1, size=(n_cases, n_mc, n_p))

            # Baseline determinacy (no noise)
            true_forces = a_mat @ w_arr
            baseline_det = float(np.mean(np.abs(true_forces) >= det_threshold))
            seed_dets.append(baseline_det)

            # Determinacy survival curve
            true_det = np.abs(true_forces) >= det_threshold
            curve = [1.0]
            for sigma in sigma_levels[1:]:
                noisy_w = np.clip(w_arr[None, None, :] + sigma * noise, 0, 1)
                noisy_forces = np.sum(noisy_w * a_mat[:, None, :], axis=2)
                noisy_det = np.abs(noisy_forces) >= det_threshold
                agree_frac = np.mean(noisy_det == true_det[:, None], axis=1)
                curve.append(float(np.mean(agree_frac >= 0.9)))

            # Find critical sigma
            crit = find_crossing(np.array(curve), sigma_levels, 0.80)
            seed_crits.append(crit)

        det_mean, det_lo, det_hi = bootstrap_ci(seed_dets)
        crit_mean, crit_lo, crit_hi = bootstrap_ci(seed_crits)
        quality = det_mean * crit_mean  # composite score

        rows.append({
            'n_principles': n_p,
            'det_mean': det_mean, 'det_lo': det_lo, 'det_hi': det_hi,
            'crit_mean': crit_mean, 'crit_lo': crit_lo, 'crit_hi': crit_hi,
            'quality': quality,
        })

        print(f"  n={n_p:2d}: determinacy={det_mean:.1%} [{det_lo:.1%}, {det_hi:.1%}]"
              f"  σ*={crit_mean:.3f} [{crit_lo:.3f}, {crit_hi:.3f}]"
              f"  quality={quality:.3f}")

    df = pd.DataFrame(rows)

    # Visualise
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Experiment 7: The Determinacy–Robustness Trade-off',
                 fontsize=13, fontweight='bold')

    # Left: Determinacy vs n
    ax1.errorbar(df['n_principles'], df['det_mean'],
                 yerr=[df['det_mean'] - df['det_lo'], df['det_hi'] - df['det_mean']],
                 marker='o', capsize=4, linewidth=2, color='steelblue')
    ax1.set_xlabel('Number of principles')
    ax1.set_ylabel('Baseline determinacy rate')
    ax1.set_title('More principles → more determinacy')
    ax1.grid(True, alpha=0.3)

    # Middle: Robustness vs n
    ax2.errorbar(df['n_principles'], df['crit_mean'],
                 yerr=[df['crit_mean'] - df['crit_lo'], df['crit_hi'] - df['crit_mean']],
                 marker='s', capsize=4, linewidth=2, color='coral')
    ax2.set_xlabel('Number of principles')
    ax2.set_ylabel('Critical weight uncertainty (σ*)')
    ax2.set_title('More principles → less robust')
    ax2.grid(True, alpha=0.3)

    # Right: Trade-off frontier (determinacy vs robustness)
    ax3.scatter(df['det_mean'], df['crit_mean'],
                c=df['n_principles'], cmap='viridis', s=120, edgecolor='black',
                zorder=5)
    for _, row in df.iterrows():
        ax3.annotate(f"n={int(row['n_principles'])}",
                     (row['det_mean'], row['crit_mean']),
                     textcoords="offset points", xytext=(8, 5), fontsize=9)
    ax3.set_xlabel('Baseline determinacy rate')
    ax3.set_ylabel('Critical weight uncertainty (σ*)')
    ax3.set_title('The trade-off frontier')
    cb = plt.colorbar(ax3.collections[0], ax=ax3)
    cb.set_label('N principles')
    ax3.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('/content/exp7_principle_count.png')
    plt.show()

    # Quantify the trade-off
    from scipy.stats import pearsonr
    r, p = pearsonr(df['det_mean'], df['crit_mean'])
    print(f"\n  Correlation (determinacy vs robustness): r={r:.3f}, p={p:.4f}")
    if r < -0.5 and p < 0.05:
        print(f"  → CONFIRMED: significant negative trade-off.")
        print(f"     Adding principles increases determinacy at the cost of robustness.")
    else:
        print(f"  → Trade-off is weak or not significant at this sample size.")

    # Find the "sweet spot" (maximum quality)
    best = df.loc[df['quality'].idxmax()]
    print(f"\n  Highest composite quality (det × σ*) at n={int(best['n_principles'])}:")
    print(f"    Determinacy: {best['det_mean']:.1%}")
    print(f"    Robustness:  ±{best['crit_mean']:.3f}")

    return df

exp7_df = experiment_7_principle_count()
exp7_df.to_csv('/content/exp7_principle_count.csv', index=False)


EXPERIMENT 7: Principle Count — Determinacy vs Robustness
  How many principles can Berman's model handle?
  n= 3: determinacy=77.3% [76.2%, 78.4%]  σ*=0.085 [0.080, 0.090]  quality=0.065
  n= 5: determinacy=81.6% [80.7%, 82.5%]  σ*=0.076 [0.072, 0.080]  quality=0.062
  n= 7: determinacy=84.2% [83.7%, 84.7%]  σ*=0.072 [0.070, 0.074]  quality=0.060
  n= 9: determinacy=86.0% [85.4%, 86.6%]  σ*=0.071 [0.068, 0.074]  quality=0.061
  n=12: determinacy=88.5% [87.8%, 89.1%]  σ*=0.075 [0.071, 0.081]  quality=0.067
  n=15: determinacy=89.5% [89.1%, 90.0%]  σ*=0.075 [0.070, 0.080]  quality=0.067
  n=20: determinacy=90.9% [90.3%, 91.5%]  σ*=0.080 [0.074, 0.087]  quality=0.073
  n=25: determinacy=91.5% [90.9%, 92.2%]  σ*=0.084 [0.078, 0.089]  quality=0.077

  Correlation (determinacy vs robustness): r=-0.077, p=0.8555
  → Trade-off is weak or not significant at this sample size.

  Highest composite quality (det × σ*) at n=25:
    Determinacy: 91.5%
    Robustness:  ±0.084


### Experiment 7 supplement — Raw Determinacy Under Noise

The main Experiment 7 measures robustness as "does the perturbed outcome agree
with the unperturbed outcome?" This supplement measures a different quantity:
"what is the raw determinacy rate when weights are noisy?" These are different
questions:

- **Agreement-based** (main): Can we trust the model's answers when weights are uncertain?
- **Raw determinacy** (supplement): Does the model still produce answers at all?

With more principles, net forces are larger (further from zero), making it
harder for noise to flip a case across the threshold. This helps the
agreement-based measure. But more principles also mean more weights to perturb,
which increases net-force variance — potentially hurting the raw determinacy
measure. If the two measures give different trade-off curves, that is itself
a finding about what "robustness" means.

In [19]:
# EXPERIMENT 7 SUPPLEMENT: RAW DETERMINACY UNDER NOISE
# ============================================================================

def experiment_7_raw_determinacy(n_seeds=20, n_cases=400, n_mc=40):
    """
    For each principle count, compute the raw determinacy rate at
    a fixed noise level (σ = 0.10, matching Experiment 1's central range),
    WITHOUT comparing to the unperturbed outcome.

    This directly measures: "If we don't know the true weights, how often
    does the model still produce a determinate answer?"
    """
    print("\n" + "=" * 70)
    print("EXPERIMENT 7 SUPPLEMENT: Raw Determinacy Under Noise")
    print("  At σ = 0.10, how often is |F_noisy| ≥ threshold?")
    print("=" * 70)

    principle_counts = [3, 5, 7, 9, 12, 15, 20, 25]
    test_sigmas = [0.0, 0.05, 0.10, 0.15, 0.20]
    det_threshold = 0.2

    rows = []
    for n_p in principle_counts:
        for sigma in test_sigmas:
            seed_dets = []
            for seed in range(n_seeds):
                rng = np.random.RandomState(seed * 100 + n_p)
                principles = make_principles(n_p, 'moderate', seed=seed)
                w_arr = np.array([p.weight for p in principles])
                a_mat = rng.uniform(-1, 1, size=(n_cases, n_p))

                if sigma == 0:
                    forces = a_mat @ w_arr
                    det_rate = float(np.mean(np.abs(forces) >= det_threshold))
                else:
                    noise = rng.normal(0, sigma, size=(n_cases, n_mc, n_p))
                    noisy_w = np.clip(w_arr[None, None, :] + noise, 0, 1)
                    noisy_forces = np.sum(noisy_w * a_mat[:, None, :], axis=2)
                    # For each case, the "MC determinacy" = fraction of samples
                    # that are determinate
                    mc_det_rate = np.mean(np.abs(noisy_forces) >= det_threshold, axis=1)
                    # Case is "determinate" if majority of MC samples say so
                    det_rate = float(np.mean(mc_det_rate >= 0.5))
                seed_dets.append(det_rate)

            m, lo, hi = bootstrap_ci(seed_dets)
            rows.append({
                'n_principles': n_p, 'sigma': sigma,
                'det_mean': m, 'det_lo': lo, 'det_hi': hi,
            })

        if sigma == 0:
            print(f"  n={n_p:2d}: baseline det={m:.1%}", end="")
        if sigma == test_sigmas[-1]:
            print(f"  at σ=0.20: det={m:.1%}")

    df = pd.DataFrame(rows)

    # Visualise: determinacy vs n_principles at different noise levels
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Experiment 7 Supplement: Raw Determinacy Under Noise',
                 fontsize=13, fontweight='bold')

    colors_s = plt.cm.plasma(np.linspace(0.1, 0.9, len(test_sigmas)))
    for j, sigma in enumerate(test_sigmas):
        sub = df[df['sigma'] == sigma]
        ax1.errorbar(sub['n_principles'], sub['det_mean'],
                     yerr=[sub['det_mean'] - sub['det_lo'],
                           sub['det_hi'] - sub['det_mean']],
                     marker='o', capsize=4, linewidth=2, color=colors_s[j],
                     label=f'σ={sigma:.2f}')
    ax1.set_xlabel('Number of principles')
    ax1.set_ylabel('Raw determinacy rate')
    ax1.set_title('Determinacy vs principle count\n(at various noise levels)')
    ax1.legend(fontsize=8)
    ax1.grid(True, alpha=0.3)

    # Right panel: determinacy DROP from baseline at each noise level
    for j, sigma in enumerate(test_sigmas[1:], 1):
        sub_noisy = df[df['sigma'] == sigma].set_index('n_principles')['det_mean']
        sub_base = df[df['sigma'] == 0].set_index('n_principles')['det_mean']
        drop = sub_base - sub_noisy
        ax2.plot(drop.index, drop.values, marker='s', linewidth=2,
                 color=colors_s[j], label=f'σ={sigma:.2f}')
    ax2.set_xlabel('Number of principles')
    ax2.set_ylabel('Determinacy drop from baseline')
    ax2.set_title('How much does noise hurt?\n(larger = worse)')
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('/content/exp7_raw_determinacy.png')
    plt.show()

    # Compare with main Experiment 7
    print(f"\n  COMPARISON: Agreement-based (main) vs Raw determinacy (supplement)")
    print(f"  {'N':>4s}  {'Agreement σ*':>14s}  {'Raw det @ σ=0.10':>18s}  {'Raw det @ σ=0.20':>18s}")
    for n_p in principle_counts:
        # Main exp7 result
        main_row = exp7_df[exp7_df['n_principles'] == n_p].iloc[0]
        # Supplement results
        raw_010 = df[(df['n_principles'] == n_p) & (df['sigma'] == 0.10)].iloc[0]
        raw_020 = df[(df['n_principles'] == n_p) & (df['sigma'] == 0.20)].iloc[0]
        print(f"  {n_p:4d}  ±{main_row['crit_mean']:.3f}          "
              f"{raw_010['det_mean']:.1%}              {raw_020['det_mean']:.1%}")

    return df

exp7_raw_df = experiment_7_raw_determinacy()
exp7_raw_df.to_csv('/content/exp7_raw_determinacy.csv', index=False)


EXPERIMENT 7 SUPPLEMENT: Raw Determinacy Under Noise
  At σ = 0.10, how often is |F_noisy| ≥ threshold?
  at σ=0.20: det=80.1%
  at σ=0.20: det=87.7%
  at σ=0.20: det=93.0%
  at σ=0.20: det=96.7%
  at σ=0.20: det=98.7%
  at σ=0.20: det=99.6%
  at σ=0.20: det=99.9%
  at σ=0.20: det=99.9%

  COMPARISON: Agreement-based (main) vs Raw determinacy (supplement)
     N    Agreement σ*    Raw det @ σ=0.10    Raw det @ σ=0.20
     3  ±0.085          77.8%              80.1%
     5  ±0.076          81.8%              87.7%
     7  ±0.072          84.8%              93.0%
     9  ±0.071          87.1%              96.7%
    12  ±0.075          89.9%              98.7%
    15  ±0.075          91.8%              99.6%
    20  ±0.080          94.7%              99.9%
    25  ±0.084          96.7%              99.9%


### Experiment 7 supplement B — Three-Way Self-Consistency (BermanModelWhy Method)

The original BermanModelWhy notebook measured robustness differently: for each
case under noise, it classified 100 MC samples into three categories (positive
force, negative force, indeterminate) and required ≥90% to agree on one
category. This is stricter than our binary agreement-with-unperturbed measure
because it distinguishes direction.

We run the same logic here to check whether the principle-count trade-off
reappears under this measurement design. If it does, the trade-off is real
but depends on the measurement choice. If it doesn't, the original finding
was an artifact of the old experiment's specific parameters.

**Measure B: Three-way self-consistency.** Under noise, do MC samples agree
on positive, negative, or indeterminate?

In [20]:
# EXPERIMENT 7 SUPPLEMENT B: THREE-WAY SELF-CONSISTENCY
# ============================================================================
# MEASURE B: Do MC samples agree on pos/neg/indet classification?

def experiment_7_threeway(n_seeds=20, n_cases=300, n_mc=60):
    """
    Replicates the BermanModelWhy measurement logic:
    For each case, run n_mc weight perturbations, classify each as
    'pos' (F > τ), 'neg' (F < -τ), or 'indet' (|F| < τ).
    Case is "robustly determinate" if ≥90% agree on one category.
    Sweep over noise levels to find where fraction drops below 80%.
    """
    print("\n" + "=" * 70)
    print("EXPERIMENT 7 SUPPLEMENT B: Three-Way Self-Consistency")
    print("  Measure B: MC samples must agree on pos/neg/indet")
    print("  (Replicates BermanModelWhy measurement design)")
    print("=" * 70)

    principle_counts = [3, 5, 7, 9, 12, 15, 20, 25]
    sigma_levels = np.linspace(0, 0.50, 16)
    det_threshold = 0.2

    rows = []
    for n_p in principle_counts:
        seed_crits = []

        for seed in range(n_seeds):
            rng = np.random.RandomState(seed * 100 + n_p)
            principles = make_principles(n_p, 'moderate', seed=seed)
            w_arr = np.array([p.weight for p in principles])

            a_mat = rng.uniform(-1, 1, size=(n_cases, n_p))
            noise = rng.normal(0, 1, size=(n_cases, n_mc, n_p))

            curve = []
            for sigma in sigma_levels:
                if sigma == 0:
                    forces = a_mat @ w_arr
                    # Three-way: what fraction of cases are clearly pos or neg?
                    frac = float(np.mean(np.abs(forces) >= det_threshold))
                    curve.append(frac)
                    continue

                # Noisy weights: (n_cases, n_mc, n_p)
                noisy_w = np.clip(w_arr[None, None, :] + sigma * noise, 0, 1)
                # Net forces: (n_cases, n_mc)
                forces = np.sum(noisy_w * a_mat[:, None, :], axis=2)

                # Three-way classification: pos, neg, indet
                pos = (forces > det_threshold).astype(float)
                neg = (forces < -det_threshold).astype(float)
                # indet = 1 - pos - neg (implicit)

                # For each case, check if ≥90% agree on one category
                frac_pos = np.mean(pos, axis=1)
                frac_neg = np.mean(neg, axis=1)
                frac_indet = 1 - frac_pos - frac_neg
                max_agreement = np.maximum(frac_pos, np.maximum(frac_neg, frac_indet))

                robust_count = float(np.mean(max_agreement >= 0.9))
                curve.append(robust_count)

            crit = find_crossing(np.array(curve), sigma_levels, 0.80)
            seed_crits.append(crit)

        m, lo, hi = bootstrap_ci(seed_crits)
        rows.append({
            'n_principles': n_p,
            'threeway_crit_mean': m, 'threeway_crit_lo': lo, 'threeway_crit_hi': hi,
        })
        print(f"  n={n_p:2d}: three-way σ* = ±{m:.3f} [{lo:.3f}, {hi:.3f}]")

    df_tw = pd.DataFrame(rows)

    # Compare all three measures side by side
    print(f"\n  COMPARISON — Three Robustness Measures:")
    print(f"  {'N':>4s}  {'A: Agreement':>14s}  {'B: Three-way':>14s}  {'Raw det@0.10':>14s}")
    for _, row in df_tw.iterrows():
        n_p = int(row['n_principles'])
        # Pull from main Exp 7
        main_row = exp7_df[exp7_df['n_principles'] == n_p]
        if len(main_row) > 0:
            agree_sigma = main_row.iloc[0]['crit_mean']
        else:
            agree_sigma = float('nan')
        # Pull from raw supplement
        raw_row = exp7_raw_df[(exp7_raw_df['n_principles'] == n_p) &
                               (exp7_raw_df['sigma'] == 0.10)]
        if len(raw_row) > 0:
            raw_det = raw_row.iloc[0]['det_mean']
        else:
            raw_det = float('nan')
        print(f"  {n_p:4d}  ±{agree_sigma:.3f}         ±{row['threeway_crit_mean']:.3f}         {raw_det:.1%}")

    # Check for trade-off under three-way measure
    from scipy.stats import pearsonr
    dets = exp7_df['det_mean'].values
    tw_crits = df_tw['threeway_crit_mean'].values
    if len(dets) == len(tw_crits):
        r, p = pearsonr(dets, tw_crits)
        print(f"\n  Correlation (determinacy vs three-way robustness): r={r:.3f}, p={p:.4f}")
        if r < -0.5 and p < 0.05:
            print(f"  → Trade-off IS present under three-way measure.")
            print(f"    More principles = more determinate but less self-consistent under noise.")
            print(f"    This reconciles with the BermanModelWhy result.")
        else:
            print(f"  → No significant trade-off under three-way measure either.")
            print(f"    The BermanModelWhy finding may have been parameter-specific.")

    # Visualise
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Three Robustness Measures Compared Across Principle Counts',
                 fontsize=13, fontweight='bold')

    ns = df_tw['n_principles'].values

    ax1.errorbar(ns, df_tw['threeway_crit_mean'],
                 yerr=[df_tw['threeway_crit_mean'] - df_tw['threeway_crit_lo'],
                       df_tw['threeway_crit_hi'] - df_tw['threeway_crit_mean']],
                 marker='D', capsize=4, linewidth=2, color='green',
                 label='B: Three-way self-consistency')
    ax1.errorbar(exp7_df['n_principles'], exp7_df['crit_mean'],
                 yerr=[exp7_df['crit_mean'] - exp7_df['crit_lo'],
                       exp7_df['crit_hi'] - exp7_df['crit_mean']],
                 marker='o', capsize=4, linewidth=2, color='steelblue',
                 label='A: Agreement with unperturbed')
    ax1.set_xlabel('Number of principles')
    ax1.set_ylabel('Critical weight uncertainty (σ*)')
    ax1.set_title('Do different measures show different trade-offs?')
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)

    # Scatter: determinacy vs each robustness measure
    ax2.scatter(exp7_df['det_mean'], exp7_df['crit_mean'],
                s=80, color='steelblue', label='A: Agreement', zorder=5)
    ax2.scatter(exp7_df['det_mean'], df_tw['threeway_crit_mean'],
                s=80, color='green', marker='D', label='B: Three-way', zorder=5)
    for j, n_p in enumerate(ns):
        ax2.annotate(f'n={n_p}',
                     (exp7_df.iloc[j]['det_mean'], df_tw.iloc[j]['threeway_crit_mean']),
                     textcoords="offset points", xytext=(6, 4), fontsize=8, color='green')
    ax2.set_xlabel('Baseline determinacy rate')
    ax2.set_ylabel('Critical uncertainty (σ*)')
    ax2.set_title('Trade-off frontier: two measures')
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('/content/exp7_threeway.png')
    plt.show()

    return df_tw

exp7_threeway_df = experiment_7_threeway()


EXPERIMENT 7 SUPPLEMENT B: Three-Way Self-Consistency
  Measure B: MC samples must agree on pos/neg/indet
  (Replicates BermanModelWhy measurement design)
  n= 3: three-way σ* = ±0.082 [0.077, 0.087]
  n= 5: three-way σ* = ±0.075 [0.071, 0.079]
  n= 7: three-way σ* = ±0.069 [0.067, 0.072]
  n= 9: three-way σ* = ±0.068 [0.065, 0.072]
  n=12: three-way σ* = ±0.073 [0.068, 0.078]
  n=15: three-way σ* = ±0.072 [0.068, 0.077]
  n=20: three-way σ* = ±0.076 [0.069, 0.084]
  n=25: three-way σ* = ±0.077 [0.072, 0.084]

  COMPARISON — Three Robustness Measures:
     N    A: Agreement    B: Three-way    Raw det@0.10
     3  ±0.085         ±0.082         77.8%
     5  ±0.076         ±0.075         81.8%
     7  ±0.072         ±0.069         84.8%
     9  ±0.071         ±0.068         87.1%
    12  ±0.075         ±0.073         89.9%
    15  ±0.075         ±0.072         91.8%
    20  ±0.080         ±0.076         94.7%
    25  ±0.084         ±0.077         96.7%

  Correlation (determinacy vs thr

## Experiment 8 — Correlated Activations

Every experiment so far draws activations independently across principles.
In practice, activations are correlated: a case involving clear statutory text
will tend to activate both textual plain meaning and original meaning; a case
raising dignity concerns will also activate equality and liberty.

This matters because correlation changes the distribution of net forces. If
principles that tend to point the same way are correlated, net forces will be
pushed further from zero (more determinacy, potentially more robust). If
principles that tend to oppose each other are correlated, the opposite happens.

We test three correlation structures:
- **Independent** (baseline): activations drawn independently.
- **Category-correlated**: textual principles correlate with each other, moral
  principles correlate with each other, but the two groups are independent.
  This is the most realistic structure.
- **Fully correlated**: all activations share a common factor (a "case
  direction"). This is an extreme case — every principle leans the same way.

For each structure, we compute the weight uncertainty tolerance and compare.

In [21]:
# EXPERIMENT 8: CORRELATED ACTIVATIONS
# ============================================================================

def experiment_8_correlated_activations(n_seeds=20, n_cases=400, n_mc=40):
    """
    Test whether activation correlation changes robustness findings.

    Three structures:
    - Independent: a_i ~ Uniform(-1,1) independently
    - Category-correlated: textual principles share a factor, moral principles
      share a different factor. Correlation within category ~0.5.
    - Fully correlated: all activations share a common factor. Correlation ~0.7.
    """
    print("\n" + "=" * 70)
    print("EXPERIMENT 8: Correlated Activations")
    print("  Does activation independence matter for the results?")
    print("=" * 70)

    sigma_levels = np.linspace(0, 0.50, 16)
    n_principles = 9
    det_threshold = 0.2

    # Define principle categories (matching Experiment 4's structure)
    # First 4: "textual/institutional", Last 5: "moral/policy"
    textual_idx = np.array([0, 1, 2, 3])
    moral_idx = np.array([4, 5, 6, 7, 8])

    def generate_activations(structure, rng, n_cases, n_p):
        """Generate activation matrix under different correlation structures."""
        if structure == 'independent':
            return rng.uniform(-1, 1, size=(n_cases, n_p))

        elif structure == 'category_correlated':
            a = np.zeros((n_cases, n_p))
            # Textual factor
            text_factor = rng.uniform(-1, 1, size=(n_cases, 1))
            a[:, textual_idx] = np.clip(
                0.5 * text_factor + 0.5 * rng.uniform(-1, 1, size=(n_cases, len(textual_idx))),
                -1, 1)
            # Moral factor
            moral_factor = rng.uniform(-1, 1, size=(n_cases, 1))
            a[:, moral_idx] = np.clip(
                0.5 * moral_factor + 0.5 * rng.uniform(-1, 1, size=(n_cases, len(moral_idx))),
                -1, 1)
            return a

        elif structure == 'fully_correlated':
            common = rng.uniform(-1, 1, size=(n_cases, 1))
            idio = rng.uniform(-1, 1, size=(n_cases, n_p))
            return np.clip(0.7 * common + 0.3 * idio, -1, 1)

    results = {}
    for structure in ['independent', 'category_correlated', 'fully_correlated']:
        all_curves = []
        baseline_dets = []

        for seed in range(n_seeds):
            rng = np.random.RandomState(seed)
            principles = make_principles(n_principles, 'moderate', seed=seed)
            w_arr = np.array([p.weight for p in principles])
            noise = rng.normal(0, 1, size=(n_cases, n_mc, n_principles))

            a_mat = generate_activations(structure, rng, n_cases, n_principles)
            true_forces = a_mat @ w_arr
            true_det = np.abs(true_forces) >= det_threshold
            baseline_dets.append(float(np.mean(true_det)))

            curve = [1.0]
            for sigma in sigma_levels[1:]:
                noisy_w = np.clip(w_arr[None, None, :] + sigma * noise, 0, 1)
                noisy_forces = np.sum(noisy_w * a_mat[:, None, :], axis=2)
                noisy_det = np.abs(noisy_forces) >= det_threshold
                agree_frac = np.mean(noisy_det == true_det[:, None], axis=1)
                curve.append(float(np.mean(agree_frac >= 0.9)))
            all_curves.append(curve)

        crits = [find_crossing(np.array(c), sigma_levels, 0.80) for c in all_curves]
        m_c, lo_c, hi_c = bootstrap_ci(crits)
        m_d, lo_d, hi_d = bootstrap_ci(baseline_dets)

        results[structure] = {
            'curves': all_curves, 'crits': crits,
            'crit_mean': m_c, 'crit_lo': lo_c, 'crit_hi': hi_c,
            'det_mean': m_d, 'det_lo': lo_d, 'det_hi': hi_d,
        }

        print(f"\n  {structure}:")
        print(f"    Baseline determinacy: {m_d:.1%} [{lo_d:.1%}, {hi_d:.1%}]")
        print(f"    Critical uncertainty: ±{m_c:.3f} [{lo_c:.3f}, {hi_c:.3f}]")

    # Visualise
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Experiment 8: Do Correlated Activations Change the Results?',
                 fontsize=13, fontweight='bold')

    colors = {'independent': 'steelblue', 'category_correlated': 'orange',
              'fully_correlated': 'green'}
    labels = {'independent': 'Independent', 'category_correlated': 'Category-correlated',
              'fully_correlated': 'Fully correlated'}

    for structure, res in results.items():
        mean_curve = np.mean(res['curves'], axis=0)
        lo_curve = np.percentile(res['curves'], 2.5, axis=0)
        hi_curve = np.percentile(res['curves'], 97.5, axis=0)
        ax1.plot(sigma_levels, mean_curve, color=colors[structure],
                 linewidth=2, label=labels[structure])
        ax1.fill_between(sigma_levels, lo_curve, hi_curve,
                         color=colors[structure], alpha=0.15)

    ax1.axhline(0.80, color='red', linestyle='--', linewidth=1.5)
    ax1.set_xlabel('Weight uncertainty (σ)')
    ax1.set_ylabel('Determinacy survival rate')
    ax1.set_title('Robustness curves by correlation structure')
    ax1.legend(fontsize=9)
    ax1.set_ylim(0.4, 1.02)
    ax1.grid(True, alpha=0.3)

    # Bar chart of critical uncertainties
    structs = list(results.keys())
    means = [results[s]['crit_mean'] for s in structs]
    errs_lo = [results[s]['crit_mean'] - results[s]['crit_lo'] for s in structs]
    errs_hi = [results[s]['crit_hi'] - results[s]['crit_mean'] for s in structs]
    bar_colors = [colors[s] for s in structs]

    ax2.bar(range(len(structs)), means, yerr=[errs_lo, errs_hi],
            color=bar_colors, alpha=0.8, edgecolor='black', capsize=5)
    ax2.set_xticks(range(len(structs)))
    ax2.set_xticklabels([labels[s] for s in structs], fontsize=9)
    ax2.set_ylabel('Critical weight uncertainty (σ*)')
    ax2.set_title('Does correlation change robustness?')
    ax2.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig('/content/exp8_correlated_activations.png')
    plt.show()

    # Statistical comparison
    from scipy.stats import mannwhitneyu
    ind_crits = results['independent']['crits']
    cat_crits = results['category_correlated']['crits']
    full_crits = results['fully_correlated']['crits']

    stat1, p1 = mannwhitneyu(ind_crits, cat_crits, alternative='two-sided')
    stat2, p2 = mannwhitneyu(ind_crits, full_crits, alternative='two-sided')

    print(f"\n  COMPARISON (Mann-Whitney U):")
    print(f"    Independent vs Category-correlated: p={p1:.4f}")
    print(f"    Independent vs Fully correlated:    p={p2:.4f}")

    if p1 > 0.05 and p2 > 0.05:
        print(f"\n  → Activation correlation does NOT significantly change robustness.")
        print(f"    Experiments 1-6 (which assume independence) remain valid.")
    elif p1 > 0.05:
        print(f"\n  → Category correlation has no effect, but full correlation does.")
        print(f"    Since category correlation is the realistic case, results hold.")
    else:
        diff = results['category_correlated']['crit_mean'] - results['independent']['crit_mean']
        direction = "more" if diff > 0 else "less"
        print(f"\n  → Category correlation makes the model {direction} robust (Δσ*={diff:+.3f}).")
        print(f"    This should be noted when interpreting Experiments 1-6.")

    return results

exp8_results = experiment_8_correlated_activations()


EXPERIMENT 8: Correlated Activations
  Does activation independence matter for the results?

  independent:
    Baseline determinacy: 85.8% [84.7%, 87.0%]
    Critical uncertainty: ±0.071 [0.066, 0.077]

  category_correlated:
    Baseline determinacy: 88.3% [87.7%, 88.9%]
    Critical uncertainty: ±0.141 [0.135, 0.148]

  fully_correlated:
    Baseline determinacy: 95.6% [95.0%, 96.1%]
    Critical uncertainty: ±0.500 [0.500, 0.500]

  COMPARISON (Mann-Whitney U):
    Independent vs Category-correlated: p=0.0000
    Independent vs Fully correlated:    p=0.0000

  → Category correlation makes the model more robust (Δσ*=+0.069).
    This should be noted when interpreting Experiments 1-6.


## Experiment 9 — Ordinal vs Cardinal Aggregation

Berman's model uses cardinal weights: dignity might be 0.70 and text might
be 0.80, and the difference between 0.70 and 0.80 matters for the net force
calculation. An alternative is ordinal aggregation: we only know that text
outranks dignity, not by how much.

The intuition favouring ordinal reasoning is that humans think in terms of
rankings ("text matters more than dignity in this area of law") rather than
precise numerical weights. If ordinal aggregation is more robust to
uncertainty, that would suggest Berman's cardinal model demands more precision
than is available in practice.

The old BermanModelWhy results showed the opposite: cardinal was *more*
robust than ordinal. This is a surprising and important finding if it
replicates — it says that the numerical precision of cardinal weights
actually helps, presumably because it gives the aggregation more information
to work with when averaging out errors.

This experiment tests both cardinal SAW and ordinal aggregation (rank-based
pseudo-weights) under the same uncertainty regime, with bootstrap CIs.

In [22]:
# EXPERIMENT 9: ORDINAL VS CARDINAL AGGREGATION
# ============================================================================

def experiment_9_ordinal_vs_cardinal(n_seeds=20, n_cases=400, n_mc=40):
    """
    Compare cardinal SAW (w_i × a_i) to ordinal aggregation
    (rank-based pseudo-weights × a_i) under weight uncertainty.

    Ordinal model: convert weights to ranks, then use normalised rank
    as the weight. This preserves ordering but discards magnitude
    information. Under uncertainty, we perturb the ORIGINAL weights
    and re-rank each time — so the ordinal model can have its rankings
    scrambled by noise.
    """
    print("\n" + "=" * 70)
    print("EXPERIMENT 9: Ordinal vs Cardinal Aggregation")
    print("  Does precise numerical weighting help or hurt under uncertainty?")
    print("=" * 70)

    sigma_levels = np.linspace(0, 0.50, 16)
    n_principles = 9
    det_threshold = 0.2

    cardinal_curves = []
    ordinal_curves = []

    for seed in range(n_seeds):
        rng = np.random.RandomState(seed)
        principles = make_principles(n_principles, 'moderate', seed=seed)
        w_arr = np.array([p.weight for p in principles])

        # Ordinal pseudo-weights: normalised ranks
        ranks = np.argsort(np.argsort(-w_arr)) + 1  # 1 = highest weight
        ordinal_w = (n_principles - ranks + 1) / n_principles

        a_mat = rng.uniform(-1, 1, size=(n_cases, n_principles))
        noise = rng.normal(0, 1, size=(n_cases, n_mc, n_principles))

        # Cardinal true outcomes
        true_forces_c = a_mat @ w_arr
        true_det_c = np.abs(true_forces_c) >= det_threshold

        # Ordinal true outcomes
        true_forces_o = a_mat @ ordinal_w
        true_det_o = np.abs(true_forces_o) >= det_threshold

        c_curve = [1.0]
        o_curve = [1.0]

        for sigma in sigma_levels[1:]:
            # Cardinal: perturb weights directly
            noisy_w_c = np.clip(w_arr[None, None, :] + sigma * noise, 0, 1)
            noisy_f_c = np.sum(noisy_w_c * a_mat[:, None, :], axis=2)
            noisy_det_c = np.abs(noisy_f_c) >= det_threshold
            agree_c = np.mean(noisy_det_c == true_det_c[:, None], axis=1)
            c_curve.append(float(np.mean(agree_c >= 0.9)))

            # Ordinal: perturb underlying weights, then RE-RANK each time
            noisy_underlying = np.clip(w_arr[None, None, :] + sigma * noise, 0, 1)
            # Re-rank for each (case, mc_sample): shape (n_cases, n_mc, n_p)
            # argsort of argsort gives ranks
            sorted_idx = np.argsort(-noisy_underlying, axis=2)
            ranks_noisy = np.argsort(sorted_idx, axis=2) + 1
            ordinal_noisy = (n_principles - ranks_noisy + 1) / n_principles

            noisy_f_o = np.sum(ordinal_noisy * a_mat[:, None, :], axis=2)
            noisy_det_o = np.abs(noisy_f_o) >= det_threshold
            agree_o = np.mean(noisy_det_o == true_det_o[:, None], axis=1)
            o_curve.append(float(np.mean(agree_o >= 0.9)))

        cardinal_curves.append(c_curve)
        ordinal_curves.append(o_curve)

        if (seed + 1) % 5 == 0:
            print(f"  seed {seed+1}/{n_seeds}")

    c_crits = [find_crossing(np.array(c), sigma_levels, 0.80) for c in cardinal_curves]
    o_crits = [find_crossing(np.array(c), sigma_levels, 0.80) for c in ordinal_curves]

    m_c, lo_c, hi_c = bootstrap_ci(c_crits)
    m_o, lo_o, hi_o = bootstrap_ci(o_crits)

    print(f"\n  Cardinal SAW: σ* = ±{m_c:.3f} [{lo_c:.3f}, {hi_c:.3f}]")
    print(f"  Ordinal:      σ* = ±{m_o:.3f} [{lo_o:.3f}, {hi_o:.3f}]")

    # Paired test
    from scipy.stats import wilcoxon
    stat, pval = wilcoxon(c_crits, o_crits)
    diff = m_c - m_o
    winner = "Cardinal" if diff > 0 else "Ordinal"
    print(f"\n  Difference: {diff:+.3f} (positive = cardinal more robust)")
    print(f"  Wilcoxon p={pval:.4f}")
    print(f"  → {winner} is {'significantly' if pval < 0.05 else 'not significantly'} more robust.")

    if m_c > m_o and pval < 0.05:
        print(f"\n  This replicates the earlier finding: cardinal precision helps.")
        print(f"  Berman's use of numerical weights is not just convenient —")
        print(f"  it makes the model more robust to uncertainty than ordinal")
        print(f"  ranking alone. The extra information in magnitudes provides")
        print(f"  a buffer against perturbation.")
    elif m_o > m_c and pval < 0.05:
        print(f"\n  Ordinal is more robust — simpler reasoning survives noise better.")
    else:
        print(f"\n  No significant difference. The choice between cardinal and")
        print(f"  ordinal aggregation is empirically neutral for robustness.")

    # Visualise
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Experiment 9: Cardinal vs Ordinal Aggregation Under Uncertainty',
                 fontsize=13, fontweight='bold')

    # Curves
    mean_c = np.mean(cardinal_curves, axis=0)
    mean_o = np.mean(ordinal_curves, axis=0)
    lo_cc = np.percentile(cardinal_curves, 2.5, axis=0)
    hi_cc = np.percentile(cardinal_curves, 97.5, axis=0)
    lo_oc = np.percentile(ordinal_curves, 2.5, axis=0)
    hi_oc = np.percentile(ordinal_curves, 97.5, axis=0)

    ax1.plot(sigma_levels, mean_c, 'b-', linewidth=2, label='Cardinal (SAW)')
    ax1.fill_between(sigma_levels, lo_cc, hi_cc, alpha=0.15, color='blue')
    ax1.plot(sigma_levels, mean_o, 'r-', linewidth=2, label='Ordinal (rank-based)')
    ax1.fill_between(sigma_levels, lo_oc, hi_oc, alpha=0.15, color='red')
    ax1.axhline(0.80, color='black', linestyle='--', linewidth=1.5)
    ax1.set_xlabel('Weight uncertainty (σ)')
    ax1.set_ylabel('Determinacy survival rate')
    ax1.set_title('Robustness curves')
    ax1.legend(fontsize=10)
    ax1.set_ylim(0.3, 1.02)
    ax1.grid(True, alpha=0.3)

    # Paired differences
    ax2.hist(np.array(c_crits) - np.array(o_crits), bins=12,
             color='mediumpurple', edgecolor='black', alpha=0.8)
    ax2.axvline(0, color='black', linewidth=1)
    ax2.axvline(diff, color='green', linewidth=2, linestyle='--',
                label=f'Mean diff = {diff:+.3f}')
    ax2.set_xlabel('σ*_cardinal − σ*_ordinal')
    ax2.set_ylabel('Count (seeds)')
    ax2.set_title(f'Paired difference (p={pval:.4f})')
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('/content/exp9_ordinal_vs_cardinal.png')
    plt.show()

    return c_crits, o_crits

exp9_c_crits, exp9_o_crits = experiment_9_ordinal_vs_cardinal()


EXPERIMENT 9: Ordinal vs Cardinal Aggregation
  Does precise numerical weighting help or hurt under uncertainty?
  seed 5/20
  seed 10/20
  seed 15/20
  seed 20/20

  Cardinal SAW: σ* = ±0.072 [0.067, 0.078]
  Ordinal:      σ* = ±0.022 [0.020, 0.023]

  Difference: +0.051 (positive = cardinal more robust)
  Wilcoxon p=0.0000
  → Cardinal is significantly more robust.

  This replicates the earlier finding: cardinal precision helps.
  Berman's use of numerical weights is not just convenient —
  it makes the model more robust to uncertainty than ordinal
  ranking alone. The extra information in magnitudes provides
  a buffer against perturbation.


## Cross-Experiment Consistency Check

Several experiments test the same configuration (n=9 principles, moderate
weights, τ=0.2) using Measure A (agreement with unperturbed). If the
notebook is internally consistent, they should give the same σ*.

In [23]:
# CROSS-EXPERIMENT CONSISTENCY CHECK
# ============================================================================

print("=" * 70)
print("CROSS-EXPERIMENT CONSISTENCY CHECK")
print("  Same config (n=9, moderate, τ=0.2), same measure (A)")
print("=" * 70)

# Collect σ* from each experiment that uses this config
consistency = {}

# Exp 1: find the cell (moderate, n=9, τ=0.20, acc=0.80)
e1_row = exp1_df[(exp1_df['distribution'] == 'moderate') &
                  (exp1_df['n_principles'] == 9) &
                  (exp1_df['det_threshold'] == 0.20) &
                  (exp1_df['accuracy_level'] == 0.80)]
if len(e1_row) > 0:
    consistency['Exp 1 (grid)'] = e1_row.iloc[0]['critical_sigma_mean']

# Exp 5 epistemic
consistency['Exp 5 (epistemic)'] = bootstrap_ci(epi_crits)[0]

# Exp 7 at n=9
e7_row = exp7_df[exp7_df['n_principles'] == 9]
if len(e7_row) > 0:
    consistency['Exp 7 (n=9)'] = e7_row.iloc[0]['crit_mean']

# Exp 8 independent
consistency['Exp 8 (independent)'] = exp8_results['independent']['crit_mean']

# Exp 9 cardinal
consistency['Exp 9 (cardinal)'] = bootstrap_ci(exp9_c_crits)[0]

print(f"\n  Experiment             σ* (Measure A)")
print(f"  {'-'*45}")
for name, val in consistency.items():
    print(f"  {name:25s} ±{val:.3f}")

vals = list(consistency.values())
spread = max(vals) - min(vals)
print(f"\n  Spread: {spread:.3f}")
print(f"  Mean:   ±{np.mean(vals):.3f}")
if spread < 0.015:
    print(f"  → Excellent internal consistency (spread < 0.015)")
elif spread < 0.030:
    print(f"  → Good internal consistency (spread < 0.030)")
else:
    print(f"  → Some inconsistency — check measurement details")

CROSS-EXPERIMENT CONSISTENCY CHECK
  Same config (n=9, moderate, τ=0.2), same measure (A)

  Experiment             σ* (Measure A)
  ---------------------------------------------
  Exp 1 (grid)              ±0.072
  Exp 5 (epistemic)         ±0.072
  Exp 7 (n=9)               ±0.071
  Exp 8 (independent)       ±0.071
  Exp 9 (cardinal)          ±0.072

  Spread: 0.001
  Mean:   ±0.072
  → Excellent internal consistency (spread < 0.015)


## Testable Criteria for Notebooks 1 and 2

The experiments above define what "good" parameters look like for Berman's SAW
model. When Notebooks 1 and 2 produce emergent weights from practice, those
weights must pass the following tests:

1. **Weight uncertainty tolerance.** Emergent weights should tolerate at least
   ±0.07 perturbation (the minimum σ* in our grid) and ideally ±0.15+ (the
   grid mean). If emergent weights are so extreme (near 0 or 1) that small
   perturbations destroy determinacy, they fail this test.

2. **Berman beats Hart.** Plugging emergent weights into the SAW model should
   still produce higher determinacy than any Hart variant. The minimum
   advantage from Experiment 4 was +19.1% (hierarchical Hart).

3. **Threshold in normatively acceptable range.** If the ABM produces an
   emergent threshold, it should fall within the τ = 0.05 to 0.30 range
   where the determinacy-stability trade-off is manageable (Experiment 6).

4. **Activation correlation helps.** Under category-correlated activations
   (Experiment 8), robustness should improve. If the ABM produces agents
   whose invocation patterns create realistic correlation structures, the
   effective robustness should be higher than the independent-activation
   baseline.

5. **Cardinal structure preserved.** Emergent weights should have meaningful
   cardinal variation (not all equal, not all binary 0/1). Experiment 9
   showed cardinal aggregation is 3.3× more robust than ordinal — this
   advantage is lost if emergent weights collapse to ranks.

These criteria are the bridge between Notebook 0 (the model works in theory)
and Notebooks 1–2 (do practice-emergent parameters satisfy the theory?).

## Summary and Interpretation

The results above establish the formal properties of Berman's SAW model
without committing to any specific parameter values. The main findings,
stated as claims that survive sensitivity analysis:

1. **Rough weights suffice — within bounds.** The model tolerates weight
   uncertainty across a wide range of configurations. The tolerance depends
   on the determinacy threshold and the number of principles, but it is
   never zero. (Experiment 1)

2. **Systematic error is the threat.** Random errors in activation assessment
   cancel out through aggregation. Systematic errors (shared methodological
   biases) do not. The ratio between their tolerances is a quantitative
   prediction about the importance of judicial diversity. (Experiment 2b)

3. **Conflicting principles coexist.** Even when principles directly oppose
   each other, weighted aggregation produces determinate outcomes at high
   rates. This is Berman's central theoretical innovation over Hart.
   (Experiment 3)

4. **Berman delivers more law than Hart.** Across multiple Hart implementations
   (from charitable to strict, including a hierarchical variant), Berman's
   aggregation model produces more determinate outcomes. The advantage is
   statistically significant and robust to the specific Hart variant.
   (Experiment 4)

5. **Aleatory uncertainty is more tolerable than epistemic uncertainty.** Judges
   can disagree about weights more than we can tolerate not knowing the weights
   at all. This supports Berman's claim that rough consensus suffices.
   (Experiment 5)

6. **The threshold is normatively loaded.** There is no "correct" threshold —
   the choice involves a quantifiable trade-off between determinacy and
   stability that depends on normative commitments. (Experiment 6)

7. **More principles means more determinacy but less robustness.** This is a
   genuine trade-off that Berman's framework must navigate. The optimal
   principle count depends on how much weight uncertainty you expect.
   (Experiment 7)

8. **Activation correlation does not undermine the findings.** Whether
   activations are independent, category-correlated, or fully correlated,
   the robustness results hold. The independence assumption in Experiments
   1–6 is not driving the results. (Experiment 8)

9. **Cardinal beats ordinal.** Berman's use of precise numerical weights is
   not just a convenience — it makes the model more robust than ordinal
   (rank-based) aggregation. The extra information in weight magnitudes
   provides a buffer against perturbation. (Experiment 9)

**What these results do not establish:** They do not tell us where the weights
or thresholds come from. Every experiment here takes parameters as given. The
question of whether parameters can emerge from practice, and whether the
emergence depends on the practice mechanism, is addressed in Notebooks 1 and 2.

In [26]:
# COMPREHENSIVE EXPORT — ALL RESULTS
# ============================================================================
# Every number in the paper traces to a file here. Nothing relies on
# "already saved" — this cell exports everything from scratch.

import json, os
out = "/content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Berman Modeling/Experiment/Computational /Outputs/0"
os.makedirs(out, exist_ok=True)
print(f"Exporting all Notebook 0 results to:\n  {out}\n")

# Recompute filtered views
df80 = exp1_df[exp1_df['accuracy_level'] == 0.80].copy()

# =====================================================================
# EXPERIMENT 1: Weight Uncertainty Tolerance
# =====================================================================
exp1_df.to_csv(f'{out}/exp1_weight_uncertainty.csv', index=False)
print("  exp1_weight_uncertainty.csv")

# =====================================================================
# EXPERIMENT 2a: Pluralist Disagreement
# =====================================================================
exp2a_df = pd.DataFrame({
    'disagreement_level': dis_levels,
    'mean_consistency': np.mean(dis_curves, axis=0),
    'lo_consistency': np.percentile(dis_curves, 2.5, axis=0),
    'hi_consistency': np.percentile(dis_curves, 97.5, axis=0),
})
exp2a_df.to_csv(f'{out}/exp2a_disagreement.csv', index=False)
print("  exp2a_disagreement.csv")

# Per-seed critical values
exp2a_crits_df = pd.DataFrame({'seed': range(len(dis_crits)), 'critical_disagreement': dis_crits})
exp2a_crits_df.to_csv(f'{out}/exp2a_critical_disagreement.csv', index=False)
print("  exp2a_critical_disagreement.csv")

# =====================================================================
# EXPERIMENT 2b: Systematic vs Random Error
# =====================================================================
err_df.to_csv(f'{out}/exp2b_error_types.csv', index=False)
print("  exp2b_error_types.csv")

# =====================================================================
# EXPERIMENT 2b Supplement: Methodology-Specific Error
# =====================================================================
exp2b_method_rows = []
for panel_name, res in exp2b_method_results.items():
    m_det, lo_det, hi_det = bootstrap_ci(res['det_crits'])
    m_dir, lo_dir, hi_dir = bootstrap_ci(res['dir_crits'])
    exp2b_method_rows.append({
        'panel_type': panel_name,
        'det_crit_mean': m_det, 'det_crit_lo': lo_det, 'det_crit_hi': hi_det,
        'dir_crit_mean': m_dir, 'dir_crit_lo': lo_dir, 'dir_crit_hi': hi_dir,
    })
pd.DataFrame(exp2b_method_rows).to_csv(f'{out}/exp2b_methodology.csv', index=False)
print("  exp2b_methodology.csv")

# Per-seed det and dir curves
for panel_name, res in exp2b_method_results.items():
    for curve_type in ['det_curves', 'dir_curves']:
        if curve_type in res:
            for seed_idx, curve in enumerate(res[curve_type]):
                pass  # curves stored in summary
    # Save per-seed crits
    crits_rows = []
    for seed_idx in range(len(res['det_crits'])):
        crits_rows.append({
            'panel_type': panel_name, 'seed': seed_idx,
            'det_crit': res['det_crits'][seed_idx],
            'dir_crit': res['dir_crits'][seed_idx],
        })
    if panel_name == list(exp2b_method_results.keys())[0]:
        all_method_crits = crits_rows
    else:
        all_method_crits.extend(crits_rows)
pd.DataFrame(all_method_crits).to_csv(f'{out}/exp2b_methodology_per_seed.csv', index=False)
print("  exp2b_methodology_per_seed.csv")

# =====================================================================
# EXPERIMENT 2c: Interval-Valued Activations
# =====================================================================
exp2c_df = pd.DataFrame({
    'range_width': rng_levels,
    'mean_consistency': np.mean(rng_curves, axis=0),
    'lo_consistency': np.percentile(rng_curves, 2.5, axis=0),
    'hi_consistency': np.percentile(rng_curves, 97.5, axis=0),
})
exp2c_df.to_csv(f'{out}/exp2c_ranges.csv', index=False)
print("  exp2c_ranges.csv")

exp2c_crits_df = pd.DataFrame({'seed': range(len(rng_crits)), 'critical_range': rng_crits})
exp2c_crits_df.to_csv(f'{out}/exp2c_critical_ranges.csv', index=False)
print("  exp2c_critical_ranges.csv")

# =====================================================================
# EXPERIMENT 3: Conflicting Principles
# =====================================================================
exp3_df.to_csv(f'{out}/exp3_conflicts.csv', index=False)
print("  exp3_conflicts.csv")

# =====================================================================
# EXPERIMENT 4: Berman vs Hart
# =====================================================================
exp4_df.to_csv(f'{out}/exp4_berman_vs_hart.csv', index=False)
print("  exp4_berman_vs_hart.csv")

# =====================================================================
# EXPERIMENT 5: Uncertainty Decomposition (Main)
# =====================================================================
# Epistemic curves
for seed_idx, curve in enumerate(epi_curves):
    if seed_idx == 0:
        epi_rows = []
    for j, val in enumerate(curve):
        epi_rows.append({'seed': seed_idx, 'sigma': np.linspace(0, 0.50, 16)[j], 'survival': val})
pd.DataFrame(epi_rows).to_csv(f'{out}/exp5_epistemic_curves.csv', index=False)
print("  exp5_epistemic_curves.csv")

# Aleatory curves
for seed_idx, curve in enumerate(ale_curves):
    if seed_idx == 0:
        ale_rows = []
    for j, val in enumerate(curve):
        ale_rows.append({'seed': seed_idx, 'sigma': np.linspace(0, 0.50, 16)[j], 'survival': val})
pd.DataFrame(ale_rows).to_csv(f'{out}/exp5_aleatory_curves.csv', index=False)
print("  exp5_aleatory_curves.csv")

# Critical values
exp5_crits = pd.DataFrame({
    'type': ['epistemic']*len(epi_crits) + ['aleatory']*len(ale_crits),
    'seed': list(range(len(epi_crits))) + list(range(len(ale_crits))),
    'critical_sigma': epi_crits + ale_crits,
})
exp5_crits.to_csv(f'{out}/exp5_epistemic_aleatory_crits.csv', index=False)
print("  exp5_epistemic_aleatory_crits.csv")

# Structural (uncalibrated)
struct_df.to_csv(f'{out}/exp5_structural.csv', index=False)
print("  exp5_structural.csv")

# =====================================================================
# EXPERIMENT 5 Supplement: Pairwise Aleatory + Calibrated Structural
# =====================================================================
exp5_pairwise = pd.DataFrame({
    'seed': range(len(exp5_ale_pairwise_crits)),
    'critical_sigma': exp5_ale_pairwise_crits,
})
exp5_pairwise.to_csv(f'{out}/exp5_aleatory_pairwise.csv', index=False)
print("  exp5_aleatory_pairwise.csv")

exp5_struct_calibrated_df.to_csv(f'{out}/exp5_structural_calibrated.csv', index=False)
print("  exp5_structural_calibrated.csv")

# =====================================================================
# EXPERIMENT 6: Normative Threshold Map
# =====================================================================
exp6_df.to_csv(f'{out}/exp6_threshold_map.csv', index=False)
print("  exp6_threshold_map.csv")

# =====================================================================
# EXPERIMENT 7: Principle Count (Main)
# =====================================================================
exp7_df.to_csv(f'{out}/exp7_principle_count.csv', index=False)
print("  exp7_principle_count.csv")

# =====================================================================
# EXPERIMENT 7 Supplement: Raw Determinacy
# =====================================================================
exp7_raw_df.to_csv(f'{out}/exp7_raw_determinacy.csv', index=False)
print("  exp7_raw_determinacy.csv")

# =====================================================================
# EXPERIMENT 7 Supplement B: Three-Way Self-Consistency
# =====================================================================
exp7_threeway_df.to_csv(f'{out}/exp7_threeway.csv', index=False)
print("  exp7_threeway.csv")

# =====================================================================
# EXPERIMENT 8: Correlated Activations
# =====================================================================
exp8_rows = []
for structure, res in exp8_results.items():
    exp8_rows.append({
        'structure': structure,
        'crit_mean': res['crit_mean'],
        'crit_lo': res.get('crit_lo', res['crit_mean']),
        'crit_hi': res.get('crit_hi', res['crit_mean']),
        'det_mean': res['det_mean'],
        'det_lo': res.get('det_lo', res['det_mean']),
        'det_hi': res.get('det_hi', res['det_mean']),
    })
pd.DataFrame(exp8_rows).to_csv(f'{out}/exp8_correlated.csv', index=False)
print("  exp8_correlated.csv")

# Per-seed critical values
exp8_seed_rows = []
for structure, res in exp8_results.items():
    for seed_idx, crit in enumerate(res['crits']):
        exp8_seed_rows.append({'structure': structure, 'seed': seed_idx, 'critical_sigma': crit})
pd.DataFrame(exp8_seed_rows).to_csv(f'{out}/exp8_correlated_per_seed.csv', index=False)
print("  exp8_correlated_per_seed.csv")

# =====================================================================
# EXPERIMENT 9: Ordinal vs Cardinal
# =====================================================================
exp9_df = pd.DataFrame({
    'seed': list(range(len(exp9_c_crits))) + list(range(len(exp9_o_crits))),
    'method': ['cardinal']*len(exp9_c_crits) + ['ordinal']*len(exp9_o_crits),
    'critical_sigma': list(exp9_c_crits) + list(exp9_o_crits),
})
exp9_df.to_csv(f'{out}/exp9_ordinal_cardinal.csv', index=False)
print("  exp9_ordinal_cardinal.csv")

# =====================================================================
# CROSS-EXPERIMENT CONSISTENCY
# =====================================================================
consistency_df = pd.DataFrame({
    'experiment': list(consistency.keys()),
    'sigma_star': list(consistency.values()),
})
consistency_df.to_csv(f'{out}/cross_experiment_consistency.csv', index=False)
print("  cross_experiment_consistency.csv")

# =====================================================================
# ALL PLOTS (re-save to output directory)
# =====================================================================
import shutil
plot_dir = f'{out}/plots'
os.makedirs(plot_dir, exist_ok=True)
for png in [f for f in os.listdir('/content') if f.endswith('.png')]:
    shutil.copy2(f'/content/{png}', f'{plot_dir}/{png}')
    print(f"  plots/{png}")

# =====================================================================
# MASTER SUMMARY JSON
# =====================================================================
summary = {
    'experiment_1': {
        'mean_critical_sigma_80pct': float(df80['critical_sigma_mean'].mean()),
        'range_min': float(df80['critical_sigma_mean'].min()),
        'range_max': float(df80['critical_sigma_mean'].max()),
        'threshold_spread': float(df80.groupby('det_threshold')['critical_sigma_mean'].mean().max() -
                                   df80.groupby('det_threshold')['critical_sigma_mean'].mean().min()),
        'distribution_spread': float(df80.groupby('distribution')['critical_sigma_mean'].mean().max() -
                                      df80.groupby('distribution')['critical_sigma_mean'].mean().min()),
        'n_principles_spread': float(df80.groupby('n_principles')['critical_sigma_mean'].mean().max() -
                                      df80.groupby('n_principles')['critical_sigma_mean'].mean().min()),
    },
    'experiment_2a': {
        'critical_disagreement_mean': float(bootstrap_ci(dis_crits)[0]),
        'critical_disagreement_lo': float(bootstrap_ci(dis_crits)[1]),
        'critical_disagreement_hi': float(bootstrap_ci(dis_crits)[2]),
    },
    'experiment_2c': {
        'critical_range_mean': float(bootstrap_ci(rng_crits)[0]),
        'critical_range_lo': float(bootstrap_ci(rng_crits)[1]),
        'critical_range_hi': float(bootstrap_ci(rng_crits)[2]),
    },
    'experiment_3': {
        'balanced_conflict_det_mean': float(exp3_df[exp3_df['case_type']=='balanced_conflict']['determinacy_rate'].mean()),
        'aligned_det_mean': float(exp3_df[exp3_df['case_type']=='aligned']['determinacy_rate'].mean()),
        'random_det_mean': float(exp3_df[exp3_df['case_type']=='random']['determinacy_rate'].mean()),
    },
    'experiment_4': {
        'berman_det': float(exp4_df['berman'].mean()),
        'hart_hierarchical_det': float(exp4_df['hart_hierarchical'].mean()),
        'advantage_hierarchical': float(exp4_df['berman'].mean() - exp4_df['hart_hierarchical'].mean()),
        'advantage_consensus_50': float(exp4_df['berman'].mean() - exp4_df['hart_consensus_50'].mean()),
        'advantage_consensus_70': float(exp4_df['berman'].mean() - exp4_df['hart_consensus_70'].mean()),
    },
    'experiment_5': {
        'epistemic_sigma_mean': float(bootstrap_ci(epi_crits)[0]),
        'aleatory_sigma_panel_mean': float(bootstrap_ci(ale_crits)[0]),
        'aleatory_sigma_pairwise_mean': float(bootstrap_ci(exp5_ale_pairwise_crits)[0]),
        'aleatory_epistemic_ratio_panel': float(bootstrap_ci(ale_crits)[0] / bootstrap_ci(epi_crits)[0]) if bootstrap_ci(epi_crits)[0] > 0 else float('nan'),
        'aleatory_epistemic_ratio_pairwise': float(bootstrap_ci(exp5_ale_pairwise_crits)[0] / bootstrap_ci(epi_crits)[0]) if bootstrap_ci(epi_crits)[0] > 0 else float('nan'),
        'structural_tf_uncalibrated': float(struct_df[struct_df['alt_model']=='threshold_filtered']['agreement'].mean()),
        'structural_tf_calibrated': float(exp5_struct_calibrated_df[exp5_struct_calibrated_df['alt_model']=='threshold_filtered']['agreement'].mean()),
    },
    'experiment_7': {
        'tradeoff_correlation': float(exp7_df['det_mean'].corr(exp7_df['crit_mean'])),
        'min_det': float(exp7_df['det_mean'].min()),
        'max_det': float(exp7_df['det_mean'].max()),
        'min_crit': float(exp7_df['crit_mean'].min()),
        'max_crit': float(exp7_df['crit_mean'].max()),
    },
    'experiment_8': {
        'independent_sigma': float(exp8_results['independent']['crit_mean']),
        'category_correlated_sigma': float(exp8_results['category_correlated']['crit_mean']),
        'fully_correlated_sigma': float(exp8_results['fully_correlated']['crit_mean']),
        'correlation_improvement_ratio': float(exp8_results['category_correlated']['crit_mean'] / exp8_results['independent']['crit_mean']) if exp8_results['independent']['crit_mean'] > 0 else float('nan'),
    },
    'experiment_9': {
        'cardinal_sigma_mean': float(bootstrap_ci(exp9_c_crits)[0]),
        'ordinal_sigma_mean': float(bootstrap_ci(exp9_o_crits)[0]),
        'cardinal_ordinal_ratio': float(bootstrap_ci(exp9_c_crits)[0] / bootstrap_ci(exp9_o_crits)[0]) if bootstrap_ci(exp9_o_crits)[0] > 0 else float('nan'),
    },
    'consistency': {
        'spread': float(max(consistency.values()) - min(consistency.values())),
        'mean_sigma': float(np.mean(list(consistency.values()))),
    },
}

with open(f'{out}/notebook0_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print("  notebook0_summary.json")

# Final count
all_files = []
for root, dirs, files in os.walk(out):
    for fname in files:
        all_files.append(os.path.join(root, fname))
print(f"\n{'='*50}")
print(f"TOTAL: {len(all_files)} files exported to {out}")
print(f"{'='*50}")

Exporting all Notebook 0 results to:
  /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Berman Modeling/Experiment/Computational /Outputs/0

  exp1_weight_uncertainty.csv
  exp2a_disagreement.csv
  exp2a_critical_disagreement.csv
  exp2b_error_types.csv
  exp2b_methodology.csv
  exp2b_methodology_per_seed.csv
  exp2c_ranges.csv
  exp2c_critical_ranges.csv
  exp3_conflicts.csv
  exp4_berman_vs_hart.csv
  exp5_epistemic_curves.csv
  exp5_aleatory_curves.csv
  exp5_epistemic_aleatory_crits.csv
  exp5_structural.csv
  exp5_aleatory_pairwise.csv
  exp5_structural_calibrated.csv
  exp6_threshold_map.csv
  exp7_principle_count.csv
  exp7_raw_determinacy.csv
  exp7_threeway.csv
  exp8_correlated.csv
  exp8_correlated_per_seed.csv
  exp9_ordinal_cardinal.csv
  cross_experiment_consistency.csv
  plots/exp5_uncertainty.png
  plots/exp7_threeway.png
  plots/exp2b_methodology_error.png
  plots/exp7_raw_determinacy.png
  plots/exp5_supplement.png
  plots/exp1_weight_uncertainty.png
  plots/exp